# RAG Pipeline: Dental Patient-Education Assistant

**Goal:** build a complete Retrieval-Augmented Generation (RAG) pipeline end-to-end, following
the pipeline architecture taught in Lab 5 (text preprocessing), Lab 6 (TF-IDF / BM25 / retrieval
metrics), Lab 7 (semantic embeddings / hybrid retrieval), Lab 8 (context building, prompt anatomy,
the full pipeline, and the final assignment tasks) and Lab 9 (grounded LLM generation and failure
analysis) — and use it to answer real patient questions grounded in a knowledge base of **19 real
dental patient-education handouts that I collected myself**.

## The Big Picture

```
Documents  ->  Chunks  ->  Retriever (TF-IDF / BM25 / Embeddings / Hybrid)
           ->  Candidate Evidence  ->  Context Building (filter, dedupe, rank, word budget)
           ->  Context Package  ->  Prompt (weak -> better -> strict)  ->  LLM Answer
```

**Knowledge base:** 19 real patient-education documents covering crowns/bridges, root canals,
implants, bone/gum grafts, extractions, scaling & root planing, dentures, Invisalign, night
guards, whitening, composite fillings and home-hygiene technique. Two topics (**crown/bridge
care** and **scaling & root planing**) each have an **older, outdated handout and a newer,
current handout**, which is used later to demonstrate conflict resolution between an outdated
and a current version of similar instructions — exactly like the `effective_date` /
`is_current` mechanic taught in Labs 8 and 9.

This notebook follows the **Final Assignment** in Lab 8 task-by-task:

| Task | What it asks for | Where it is done in this notebook |
|---|---|---|
| Task 1 | Build your own corpus (>=15 docs, metadata, 2+ outdated, 3+ paraphrase traps) | Section 1 |
| Task 2 | >=10 queries + ground truth, 4+ paraphrased | Section 4 |
| Task 3 | TF-IDF / Embedding / Hybrid retrievers, evaluated, 3+ alpha values | Sections 2-3, 5 |
| Task 4 | Context package for >=5 queries (filter outdated, dedupe, 150-word budget, CURRENT/OUTDATED labels) | Section 6 |
| Task 5 | Weak / better / strict prompts + wrong-choice scenario for each | Section 7 |
| Task 6 | Error analysis for >=3 failed queries | Section 9 |

## Section 0 — Install & Import Libraries

Same stack used throughout Labs 6-9: `pandas` / `numpy` for tables and math, `scikit-learn` for
TF-IDF and cosine similarity, `rank-bm25` for BM25 (bonus retriever, taught in Lab 6/7), and
`sentence-transformers` for real semantic embeddings (`all-MiniLM-L6-v2`, exactly as required by
Task 3). The Lab 5 preprocessing profile is reimplemented here with plain `re` (no NLTK download
needed) since only lowercasing/whitespace/URL cleanup is used, per the `minimal_clean` profile.

In [17]:
!pip install -q rank-bm25 sentence-transformers

In [18]:
import re
import string
import os
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_columns', 50)

## Section 1 — Knowledge Base Setup (Task 1)

Each document carries the exact metadata schema used in Lab 8 / Lab 9's shared corpus:
`document_id`, `title`, `doc_type`, `effective_date`, `is_current`, `text`.

**Two outdated documents that conflict with current ones (Task 1 requirement: at least 2):**
- `D1` *"Post-Operative Instructions for Crown or Bridges"* (2021, `is_current=False`) is
  superseded by `D9` *"Crown or Bridge Post-Operative Care (Updated)"* (2025, `is_current=True`).
  Both share `doc_type="crown_bridge_care"`.
- `D2` *"Root Planing and Scaling for Gum Disease"* (2020, `is_current=False`) is superseded by
  `D12` *"Deep Cleaning (Scaling and Root Planing) Post-Treatment Instructions"* (2025,
  `is_current=True`). Both share `doc_type="periodontal_scaling"`.

**At least 3 paraphrase traps (query wording differs from the document's own wording)** are built
into the query set in Section 4, for example:
1. Queries say *"gum disease"* while the current document (`D12`) only says *"periodontal
   disease"* — the outdated document (`D2`) actually says *"gum disease"* explicitly, so a purely
   lexical retriever can be fooled into preferring the outdated document.
2. Queries say *"can't open my mouth all the way"* while the documents use the clinical term
   *"Trismus (stiffness)"*.
3. Queries say *"having a tooth pulled"* while the documents say *"tooth extraction"*.
4. Queries say *"missing spots when I brush"* while the document says *"areas you might be
   missing"* / *"missed brushing"*.

In [19]:
DOCUMENTS = [{'document_id': 'D1',
  'title': 'Post-Operative Instructions for Crown or Bridges',
  'doc_type': 'crown_bridge_care',
  'effective_date': '2021-04-01',
  'is_current': False,
  'text': 'Post-Operative Instructions for Crown or Bridges\n'
          'Congratulations! You now have you final restoration which is permanent. Please\n'
          'remember that your new crown or bridge, as well as the rest of your natural teeth, '
          'need\n'
          'regular maintenance with your doctor and hygienist. This is the only way to keep them\n'
          'healthy.\n'
          'Routine cleaning and maintenance appointments are recommended every 3 to 6\n'
          'months, depending on the condition of your gums. Please make an appointment if you\n'
          'have not already done so.\n'
          '- Your gums may have become sore during this procedure. Please rinse with warm\n'
          'salt water a few times daily. Flossing is very important to help heal the area. Even\n'
          'if it is sore, please attempt to floss daily, as this will help it to heal.\n'
          '- Bridges need floss threaders. Threaders allow you to floss below the bridge\n'
          'properly. We will show you how to use them at your final appointment. Threaders\n'
          'may be purchased at any drug store.\n'
          '- If a permanent crown or bridge feels "high" or like you are hitting that tooth '
          'first,\n'
          'please call the office to have it adjusted. This restoration WILL NOT adjust itself\n'
          'like a temporary might. Without correction you will have a toothache and/or a\n'
          'sore jaw.\n'
          '- Avoid chewing sticky, chewy foods, and chewing gum for 24-hours.'},
 {'document_id': 'D2',
  'title': 'Root Planing and Scaling for Gum Disease',
  'doc_type': 'periodontal_scaling',
  'effective_date': '2020-09-01',
  'is_current': False,
  'text': 'Root Planning and Scaling for Gum Disease\n'
          'Root planning and scaling is one of the most effective ways to treat gum disease before '
          'it\n'
          'becomes severe. Root planing\n'
          'and scaling cleans between the gums and the teeth down to the roots. Your dentist may\n'
          'need to use a local anesthetic to\n'
          'numb your gums and the roots of your teeth.\n'
          'The Drs. may place antibiotic fibers into the pockets between your teeth and gums. The\n'
          'antibiotic will he help speed\n'
          'healing and prevent infection.\n'
          'What to Expect After Treatment?\n'
          'If anesthesia is used, your lips and gums may remain numb for a few hours. Planing and\n'
          'scaling causes little to no\n'
          'discomfort.\n'
          'Why It Is Done?\n'
          'Root planing and scaling is done when gums have either started to pull away from the '
          'teeth\n'
          'or the roots of the teeth have\n'
          'hard mineral deposits (tartar) on them.\n'
          'How Well It Works?\n'
          'If you maintain good dental care after the procedure, the progression of gum disease\n'
          'should stop: And your gums will heal\n'
          'and become firm and pink again.\n'
          'Risk?\n'
          'Root planing and scaling can introduce harmful bacteria into the bloodstream. Gum '
          'tissue\n'
          'is also at risk of infection. You\n'
          'may need to take antibiotics before and after surgery if you have a condition that puts '
          'you\n'
          'at high risk for a severe\n'
          'infection of if infections are particularly dangerous for you. You may need to take\n'
          'antibiotics if you:\n'
          'Have certain heart problems that make it dangerous for you to get a heart infection '
          'called\n'
          'endocarditis.\n'
          'Have an impaired immune system\n'
          'Had recent major surgeries or have man-made body parts, such as an artificial hip or '
          'heart\n'
          'valve.\n'
          'What to Think About?\n'
          'Root planing and scaling is a simple procedure that can work very well to stop gum\n'
          'disease.\n'
          'Brush and floss regularly afterward. Without proper dental care, your gum disease may\n'
          'progress.\n'
          'To promote healing, stop all use of tobacco. Smoking or using spit tobacco reduces '
          'your\n'
          'ability to fight infection\n'
          'of your gums and delays healing.\n'
          'Further prolonging treatment can result in loss of teeth, server pain, infection, as '
          'well as\n'
          'additional cost in dental\n'
          'treatment.'},
 {'document_id': 'D3',
  'title': 'Post-Operative Instructions Following Oral Surgery',
  'doc_type': 'oral_surgery_general',
  'effective_date': '2025-01-10',
  'is_current': True,
  'text': 'Post-Operative Instructions Following Oral Surgery\n'
          'Gauze has been placed into the extraction area to allow you to apply pressure and '
          'prevent\n'
          'excessive bleeding. You should bite onto gauze for 30 minutes, after this time has '
          'passed\n'
          'remove the gauze and if the site is still bleeding replace with new gauze for another '
          '30\n'
          'minutes. If this does not allow the bleeding to subside you can try biting onto a damp '
          'tea\n'
          'bag (the acids in the tea bag have a clotting effect). Do not agitate the area by '
          'removing and\n'
          'replacing the gauze often, this can cause the site to bleed excessively.\n'
          'Some patients will receive sutures in the area of the extraction/surgery site to assist '
          'with\n'
          'healing. If you have sutures the assistant will advise you if they are dissolvable or '
          'if you will\n'
          'require an appointment to have your non-dissolvable sutures removed.\n'
          'It is possible that you may have some swelling and bruising in the general area of the\n'
          'extraction/surgery site. This will usually reach its peak on day 2 or 3 after surgery. '
          'It will\n'
          'gradually disappear and is of no cause for major concern unless the swelling is '
          'obstructing\n'
          'your airway. To treat the swelling, you will apply an ice pack wrapped in a towel for '
          '10\n'
          'minutes on and 10 minutes off for up to two (2) hours at a time.\n'
          'Additional Instructions\n'
          '- Smoking should be avoided for the first 48 hours to avoid getting a dry socket.\n'
          '- Do not use a Straw to drink for the first 48 hours.\n'
          '- Do not spit forcefully as this may dislodge the clot that is protecting your\n'
          'extraction/surgery site.\n'
          '- Do not participate in any strenuous physical activity for a minimum of 24 hours '
          'after\n'
          'surgery.\n'
          '- If you have had a tooth extracted from the upper, refrain from blowing your nose for\n'
          'the first 48 hours.\n'
          '- Keep your fingers, tongue and toothbrush away from the extraction/surgery site.\n'
          '- Do not rinse your mouth for the first 24 hours, mouthwashes like Scope or Listerine\n'
          'should be avoided until the area has healed in 24 hours you will begin frequently and\n'
          'gently rinsing with a warm salt water. (2tsp Salt dissolved into 8oz of water)\n'
          '- You may have cold or lukewarm but not HOT liquids for the first 4-6 hours following\n'
          'surgery, after this time you may incorporate soft foods. Do not drink any carbonated\n'
          'beverages for the first 48 hours to prevent dislodging the clot that is protecting the\n'
          'site. Also avoid crunchy foods that may cause particles to lodge into the\n'
          'extraction/surgical site.'},
 {'document_id': 'D4',
  'title': 'Post-op Instructions for Implants and Extractions',
  'doc_type': 'implant_extraction_general',
  'effective_date': '2025-02-05',
  'is_current': True,
  'text': 'Post-op Instructions for Implants and Extractions\n'
          'WHEN SHOULD YOU NOTIFY THE DOCTOR\n'
          '1. If profuse bleeding continues after 3-4 hours of applied pressure.\n'
          '2. If you are unable to maintain a nutritious diet after 48hours.\n'
          '3. If the pain and/or swelling increases after the third day.\n'
          '4. If an oral bandage becomes dislodged prior to the third day.\n'
          '5. If you have an allergic reaction to medications such as: a. Skin rash b. Hives c.\n'
          'Elevated temperature d. Increased and/or erratic heart rate e. Nausea/vomiting f.\n'
          'Dizziness/fainting g. Blurred vision\n'
          'GENERAL INSTRUCTIONS\n'
          '- DO NOT rinse for at least 24hours after the surgery\n'
          '- DO NOT exercise or do heavy lifting for 3 to 5days after surgery\n'
          '- DO NOT smoke for at least 72 hours\n'
          'WHAT TO EXPECT FOLLOWING ANY SURGERY\n'
          'BLEEDING To slow or prevent bleeding, bite with light pressure on the gauze pack that '
          'has\n'
          'been placed over the surgical area. Pressure should be applied in 20 to 30 minute '
          'intervals\n'
          'and repeated until the bleeding is brought under control. If bleeding persists without\n'
          'slowing for several hours, apply a gauze soaked in strong tea and repeat the above '
          'steps\n'
          'until the bleeding stops. Exercising and heavy lifting will raise your blood pressure '
          'and will\n'
          'dislodge the blood clot and bleeding will resume. Avoid exercising for 3 to 5 days '
          'following\n'
          'surgery.\n'
          'SWELLING To prevent and/or minimize swelling, apply ice packs at 10minute intervals to\n'
          'the surgical area. After 72 hours, apply warm compresses to the area to relieve '
          'swelling.\n'
          'Swelling is a natural part of the healing process and can be expected for 3days to '
          'several\n'
          'weeks depending on the nature and the extent of the surgery.\n'
          'DISCOMFORT Following most surgical procedures there may or may not be pain,\n'
          'depending on your threshold for pain. You will be provided with medication for '
          'discomfort\n'
          'that is appropriate for you. In most cases, a non-narcotic pain regimen will be given\n'
          'consisting of Acetaminophen (Tylenol) & Ibuprofen (Motrin). These two medications, '
          'taken\n'
          'together, will be as effective as a narcotic without any of the side effects associated '
          'with\n'
          'narcotics. If a narcotic has been prescribed, follow the directions carefully. If you '
          'have any\n'
          'questions about these medications interacting with other medications you are presently\n'
          'taking, please call our office first, your physician and/or your pharmacist.'},
 {'document_id': 'D5',
  'title': 'Post-Operative Instructions for Root Canal Treatment',
  'doc_type': 'root_canal',
  'effective_date': '2025-03-01',
  'is_current': True,
  'text': 'Post-Operative Instructions for Root Canal Treatment\n'
          'Endodontic, or root canal, treatment is now complete. The canals inside the roots have\n'
          'been cleaned, irrigated, medicated, and permanently sealed. The opening in the tooth '
          'with\n'
          'the root canal treatment has been sealed with a filling.\n'
          'Delay in obtaining the final restoration (crown) on this tooth may result in the '
          'fracture\n'
          'and/or possible loss of this tooth.\n'
          'What to expect:\n'
          '- It is NOT uncommon for a tooth to be uncomfortable or even exhibit a dull ache\n'
          'immediately after receiving root canal therapy. This should subside within a few\n'
          'days or weeks. This occurs because of conditions, which exist before treatment was\n'
          'started. Experience shows that if there was pain prior to treatment, there may be\n'
          'some degree of pain that will continue following the treatment. Remember that pain\n'
          'radiates. You may feel discomfort in other teeth in the area. This can be created by\n'
          'inflammation and/or increase in blood volume to occur in the healing process.\n'
          '- Your tooth will be sensitive to biting pressure and may even appear to feel loose.\n'
          'This feeling is a result of the nerve ending just outside the end of the root of the\n'
          'treated tooth. Discomfort in the area for a few days to weeks is common. Warm salt\n'
          'water rinses for the next two days will help. Try to avoid chewing on this area until\n'
          'the tenderness has subsided.\n'
          '- Occasionally a small "bubble" or "pimple" will appear on the gum tissue within a few\n'
          'days after completion of a root canal. This is a release of pressure and bacteria\n'
          'which is no longer sustained around the tooth. This should disappear within a few\n'
          'days.\n'
          '- The office staff will review with you the timeline and steps needed to complete the\n'
          'final restoration (crown) for this tooth.'},
 {'document_id': 'D6',
  'title': 'Opalescence Take-Home Whitening Step-by-Step Instructions',
  'doc_type': 'whitening_take_home_opalescence',
  'effective_date': '2025-04-01',
  'is_current': True,
  'text': 'Opalescence Step-by-Step Instructions\n'
          'Step 1: Spread gel into your custom bleaching tray/retainer using a maximum of '
          'one-half\n'
          '(1/2) to one-third (1/3) of a syringe.\n'
          'Step 2: Brush your teeth, and then insert both trays (top and bottom) onto your teeth.\n'
          'Optionally you can do the process with one tray at a time or alternate treatments '
          'between\n'
          'the top and bottom.\n'
          'Any excess gel that overflows from the tray onto the gums should be removed with a '
          'cotton\n'
          'ball, tissue, soft toothbrush, or clean finger. This is important because prolonged '
          'exposure\n'
          'to the gums can cause significant irritation.\n'
          'Opalescence instructions state that this process can be repeated every day for the\n'
          'duration of the treatment but we recommend at least every-other day to reduce pain and\n'
          'sensitivity by giving your teeth and gums time to recover. There is really no rush and '
          'spacing\n'
          'out the treatment will provide greater comfort without and negative effects.\n'
          'Step 3: For Opalescence 20%, remove the tray after 2 to 4 hours during the day and '
          'only\n'
          'consider overnight treatment if your teeth can tolerate the 20% concentration well.\n'
          'Although the standard Opalescence instructions do not include this, based on customer\n'
          'feedback, we do not recommend an overnight exposure for the first few treatments until\n'
          'you have tried shorter durations and found that your tooth and gum sensitivity is '
          'tolerable.\n'
          'Step 4: After removing the trays, brush your teeth normally. Then rinse the trays in '
          'cool\n'
          'water (note that hot water can warp or distort certain types of bleaching trays) and '
          'store\n'
          'them in a cool place out of the sun.\n'
          'Length of Treatment\n'
          'This is a common question: how long should you continue to use Opalescence. In fact\n'
          'there is nothing in the Opalescence instructions about the duration. Your dentist '
          'might\n'
          'have come up with an estimate but really it is up to you. If the whitening gel is '
          'working for\n'
          'you, as long as the sensitivity or irritation are tolerable or negligible, you can use '
          'this\n'
          'product until you achieve the desired whiteness. In short, here is the only '
          'Opalescence\n'
          'instructions we can give on duration: continue to use it until you achieve the desired\n'
          'results.\n'
          'After you have completed the treatment, you will probably want to do a touch-up '
          'treatment\n'
          'every so often to maintain your pearly whites. For example, once every 6 months you '
          'might\n'
          'want to resume the treatment for several days but again. However, there are no\n'
          "Opalescence instructions on this point; everyone's teeth are different due to many "
          'factors\n'
          'so it is up to you and/or your dentist to find how Opalescence works best for you.'},
 {'document_id': 'D7',
  'title': 'Post-Operative Instructions After Bone Graft Surgery',
  'doc_type': 'bone_graft',
  'effective_date': '2025-01-20',
  'is_current': True,
  'text': 'BONE GRAFT\n'
          '- MOST IMPORTANT DIRECTION: You have been given an IRRIGATION SYRINGE and\n'
          'instructed how and when to use it. You may begin gentle rinsing with IRRIGATION\n'
          'SYRINGE using a saltwater solution (1 or 2 teaspoon salt + 8 ounces warm water).\n'
          'Be sure to RINSE AFTER EACH MEAL OR SNACK, or every 2-3 hours! Refill you\n'
          'syringe 2-3 times and rinse gently until no more food is seating at the site. If you\n'
          'keep your surgical site clean, you will heal, if you do not, it will cause secondary\n'
          'infection. Remember, it takes only one time for food to get into the surgical site and\n'
          'then secondary infection will start, so be sure to RINSE AFTER EACH MEAL OR\n'
          'SNACK!\n'
          '- CHLORHEXIDINE RINSING: In addition to warm salt water rinses, you would also\n'
          'need to rinse with “liquid antibiotic” called Chlorhexidine. If you been given\n'
          'prescription at your appointment, then follow these instructions. After breakfast\n'
          'and dinner, or morning and evening, when you rinse with warm salt water and get\n'
          'the site cleaned, now you need to rinse with Chlorhexidine, use same syringe and do\n'
          'exactly the same step as with warm salt water, but now use Chlorhexidine.\n'
          '- ANTIBIOTICS: Take all of them as PRESCRIBED AND DIRECTED to you until you\n'
          'finish all of your antibiotics. Women: some antibiotics can reduce the effectiveness\n'
          'of birth control pills. Use alternate birth control methods for two months.\n'
          '- PAIN: Some discomfort is normal after surgery. To minimize pain, take Tylenol,\n'
          'Motrin or Advil, or similar non-aspirin pain reliever to maintain comfort. Start take '
          'it\n'
          'before the anesthesia wears off. If prescription pain medication is prescribed, take '
          'it\n'
          'as instructed on the label. Don’t exceed the dose on the label. Taking it with food\n'
          'will help reduce upset stomach. Avoid driving or operating heavy machinery when\n'
          'taking pain prescriptions. Do not drink alcohol while taking prescription pain\n'
          'medications. Strong pain medications, like Norco, must be only taken before\n'
          'bedtime.\n'
          '- DO NOT: Swish, suck through a straw, spit or smoke, it can all dislodge the clot.\n'
          'Keep anything sharp from entering the wound (crunchy food, toothpicks, eating\n'
          'utensils). Be sure to chew on the opposite side until the area heals\n'
          'completely. Understand that any food caught inside the extraction socket (place\n'
          'where tooth used to be) can cause infection, pain and additional surgery, so stay on\n'
          'soft or liquid food diet and give your body enough time to heal, BUT BE SURE TO\n'
          'RINSE AFTER EACH MEAL OR SNACK!\n'
          '- BLEEDING: When you leave the office, you might be biting on a gauze pad to control\n'
          'bleeding. Keep slight pressure on this gauze for at least 30 to 60 minutes. Don’t\n'
          'change it during this time; it needs to remain undisturbed while a clot forms in the\n'
          'extraction socket. After 30 minutes you may remove it. You may bite on another\n'
          'gauze or a tea bag for another 30 minutes if you feel it is still bleeding. Small\n'
          'amounts of blood in the saliva can make your saliva appear quite red. This is normal\n'
          'and may be noticed the rest of the day after the procedure.\n'
          '- DIET: Eat soft foods until the area is healed. Maintain a good, balanced diet, you\n'
          'need proper nutrition to heal. Drink plenty of water. Avoid alcohol for 48\n'
          'hours. Keeping the area clean from food debris will promote good healing.\n'
          '- SMOKING: Smoking should be stopped following surgery. Healing and success of\n'
          'the surgery will be substantially reduced by the cigarette smoke chemicals in your\n'
          'body. Also the suction created when inhaling cigarettes can dislodge the clot.\n'
          'Smokers are at greater risk of developing a painful condition called “Dry Socket”,\n'
          'which can hurt worse then a toothache and there is no cure for it. You must quit for\n'
          'at least 5 day to allow initial healing.\n'
          '- NAUSEA: This is most often caused by taking pain medications on an empty\n'
          'stomach. Reduce nausea by preceding each pain pill with soft food, and taking the\n'
          'pill with a large glass of water.\n'
          '- SWELLING: Applying an ice bag to the face over the operated area will minimize\n'
          'swelling. Apply for 15 minutes, then remove for 15 minutes. Continue this for the\n'
          'first day.\n'
          '- NUMBNESS: The local anesthetic will cause you to be numb for several hours after\n'
          'you leave the office. Be very careful not to bite, chew, pinch, or scratch the numb\n'
          'area. Sometimes the extraction causes residual numbness or tingling for six weeks\n'
          'or longer. Contact our office if you experience these symptoms.\n'
          '- BRUSHING: Do not brush your teeth for the first 8 hours after surgery. After this, '
          'you\n'
          'must brush your teeth to reduce bacteria amount, but avoid the area of surgery for\n'
          'few days.\n'
          '- ACTIVITY: After leaving the office, rest and avoid strenuous activities for the\n'
          'remainder of the day. Keeping blood pressure lower will reduce bleeding and aid\n'
          'healing.\n'
          '- SINUS: If your sinus was involved in the procedure, you should avoid blowing your\n'
          'nose or playing a wind musical instrument for one week. Use of decongestant\n'
          'medications might be recommended.\n'
          '- SUTURES: If you had resorbable sutures placed, then they will dissolve on its own\n'
          'in 7-10 days and you do not need to worry about suture removal. If you have sutures\n'
          'that do not resorb, then you need to return back to the office for a sutures removal.\n'
          '- SPECIAL CONSIDERATIONS – Trismus (stiffness) in the face muscles may cause\n'
          'difficulty in opening your mouth for a period of days. Moist heat compresses can\n'
          'minimize this condition. You may experience aching from other teeth. This\n'
          'discomfort is caused by referred pain and is a temporary condition. It is not unusual\n'
          'to develop bruising in the area of the extraction. There may be a slight elevation in\n'
          'temperature for 24-48 hours. If the fever persists, please contact our office.\n'
          '- FOLLOW-UP APPOINTMENTS: You WILL NEED TO RETURN to the office for a brief\n'
          'follow-up healing check.\n'
          'Following these instructions very closely will greatly help your comfort, and promote\n'
          'uneventful healing of the area. If any of the instructions are not followed, you might '
          'have\n'
          'significantly more discomfort, and the success of the procedure may be affected.'},
 {'document_id': 'D8',
  'title': 'Brushing With a Disclosing Agent (Home Hygiene Guide)',
  'doc_type': 'home_hygiene_technique',
  'effective_date': '2025-05-01',
  'is_current': True,
  'text': 'BRUSHING\n'
          'Brushing your teeth with a disclosing agent can be a great way to visualize areas you '
          'might\n'
          "be missing during your regular routine. Here's how to do it effectively and safely:\n"
          'Before you begin:\n'
          "- Gather your supplies: You'll need your regular toothbrush, toothpaste, a disclosing\n"
          'tablet or rinse (click here for list of products on Amazon), and a glass of water.\n'
          "- Choose your lighting: Good lighting is key for seeing the disclosing agent's "
          'effects.\n'
          'Natural daylight or a well-lit bathroom are ideal.\n'
          'Brushing with the disclosing agent:\n'
          '1. Brush as usual: Start by brushing your teeth with your regular toothpaste for 2\n'
          'minutes, focusing on all surfaces of each tooth.\n'
          '2. Use the disclosing agent: After rinsing with water, chew the disclosing tablet or\n'
          'swish the rinse in your mouth for 30 seconds, coating all your teeth.\n'
          '3. Rinse lightly: Rinse with a small amount of water, leaving some of the disclosing\n'
          'agent on your teeth. This will highlight areas you missed brushing.\n'
          'Analyzing the results:\n'
          '- Look for stained areas: The disclosing agent will stain areas where plaque and food\n'
          'debris remain, appearing as pink, red, or purple spots.\n'
          '- Focus on missed areas: Pay particular attention to the gum line, behind teeth, and\n'
          'in between molars, where plaque often builds up.\n'
          '- Refine your brushing: Use your toothbrush to carefully remove the stained areas,\n'
          'focusing on gentle circular motions and short back-and-forth strokes.\n'
          'Additional tips:\n'
          "- Don't swallow the disclosing agent: It's meant for temporary staining, not\n"
          'ingestion.\n'
          '- Rinse thoroughly after: Remove all traces of the disclosing agent with water to\n'
          'avoid staining your tongue or lips.\n'
          '- Brush regularly: Using a disclosing agent once or twice a week can help you\n'
          'develop good brushing habits and maintain optimal oral hygiene.\n'
          'Remember, even with good brushing, plaque and tartar can build up over time, so '
          'regular\n'
          'dental cleanings are still essential for maintaining healthy teeth and gums.'},
 {'document_id': 'D9',
  'title': 'Crown or Bridge Post-Operative Care (Updated)',
  'doc_type': 'crown_bridge_care',
  'effective_date': '2025-08-01',
  'is_current': True,
  'text': 'CROWN OR BRIDGE\n'
          '- DO NOT DISTURB THE AREA: For the next few days, and especially the first 24\n'
          'hours, it is very important to allow your body to form a good clot and start natural\n'
          'healing process.\n'
          '- DO NOT: Swish, suck through a straw, spit or smoke, it can all dislodge the\n'
          'clot. Keep anything sharp from entering the wound (crunchy food, toothpicks, eating\n'
          'utensils).\n'
          '- Be sure to chew on the opposite side until the area heals completely.\n'
          '- Understand that any food caught inside the extraction socket (place where tooth\n'
          'used to be) can cause infection, pain and additional surgery, so stay on soft or '
          'liquid\n'
          'food diet and give your body enough time to heal.\n'
          '- MOST IMPORTANT DIRECTION: You have been given an IRRIGATION SYRINGE and\n'
          'instructed how and when to use it. You may begin gentle rinsing with IRRIGATION\n'
          'SYRINGE using a saltwater solution (1 or 2 teaspoon salt + 8 ounces warm water).\n'
          'Be sure to RINSE AFTER EACH MEAL OR SNACK! If you keep your surgical site clean,\n'
          'you will heal, if you do not, it will cause secondary infection. Remember, it takes\n'
          'only one time for food to get into the surgical site and then secondary infection will\n'
          'start, so be sure to RINSE AFTER EACH MEAL OR SNACK!\n'
          '- BLEEDING: When you leave the office, you might be biting on a gauze pad to control\n'
          'bleeding. Keep slight pressure on this gauze for at least 30 to 60 minutes. Don’t\n'
          'change it during this time; it needs to remain undisturbed while a clot forms in the\n'
          'extraction socket. After 30 minutes you may remove it. You may bite on another\n'
          'gauze or a tea bag for another 30 minutes if you feel it is still bleeding. Small\n'
          'amounts of blood in the saliva can make your saliva appear quite red. This is normal\n'
          'and may be noticed the rest of the day after the procedure.\n'
          '- DIET: Eat soft foods until the area is healed. Maintain a good, balanced diet, you\n'
          'need proper nutrition to heal. Drink plenty of water. Avoid alcohol for 48\n'
          'hours. Keeping the area clean from food debris will promote good healing.\n'
          '- SMOKING: Smoking should be stopped following surgery. Healing and success of\n'
          'the surgery will be substantially reduced by the cigarette smoke chemicals in your\n'
          'body. Also the suction created when inhaling cigarettes can dislodge the clot.\n'
          'Smokers are at greater risk of developing a painful condition called “Dry Socket”,\n'
          'which can hurt worse then a toothache and there is no cure for it. You must quit for\n'
          'at least 5 day to allow initial healing.\n'
          '- PAIN: Some discomfort is normal after surgery. To minimize pain, take Tylenol,\n'
          'Motrin or Advil, or similar non-aspirin pain reliever to maintain comfort. Start take '
          'it\n'
          'before the anesthesia wears off. If prescription pain medication is prescribed, take '
          'it\n'
          'as instructed on the label. Don’t exceed the dose on the label. Taking with food or\n'
          'milk will help reduce upset stomach. Avoid driving or operating heavy machinery\n'
          'when taking pain prescriptions. Do not drink alcohol while taking prescription pain\n'
          'medications. Strong pain medications, like Norco, must be only taken before\n'
          'bedtime.\n'
          '- NAUSEA: This is most often caused by taking pain medications on an empty\n'
          'stomach. Reduce nausea by preceding each pain pill with soft food, and taking the\n'
          'pill with a large glass of water.\n'
          '- SWELLING: Applying an ice bag to the face over the operated area will minimize\n'
          'swelling. Apply for 15 minutes, then remove for 15 minutes. Continue this for the\n'
          'first day.\n'
          '- NUMBNESS: The local anesthetic will cause you to be numb for several hours after\n'
          'you leave the office. Be very careful not to bite, chew, pinch, or scratch the numb\n'
          'area. Sometimes the extraction causes residual numbness or tingling for six weeks\n'
          'or longer. Contact our office if you experience these symptoms.\n'
          '- BRUSHING: Do not brush your teeth for the first 8 hours after surgery. After this, '
          'you\n'
          'must brush your teeth to reduce bacteria amount, but avoid the area of surgery for\n'
          'few days.\n'
          '- RINSING: Avoid all rinsing or swishing for 24 hours after extraction. Rinsing can\n'
          'disturb the formation of a healing blood clot which is essential to proper healing.\n'
          'This could cause bleeding and risk of dry socket. After 24 hours you may begin\n'
          'gentle rinsing with a saltwater solution (1 or 2 teaspoon salt + 8 ounces warm water)\n'
          'after each meal or snack. Avoid commercial mouth rinses that contain alcohol.\n'
          '- ACTIVITY: After leaving the office, rest and avoid strenuous activities for the\n'
          'remainder of the day. Keeping blood pressure lower will reduce bleeding and aid\n'
          'healing.\n'
          '- ANTIBIOTICS: If you were given an antibiotic prescription, take all of them as\n'
          'directed until they are gone. Women: some antibiotics can reduce the effectiveness\n'
          'of birth control pills. Use alternate birth control methods for two months.\n'
          '- SINUS: If your sinus was involved in the procedure, you should avoid blowing your\n'
          'nose or playing a wind musical instrument for one week. Use of decongestant\n'
          'medications might be recommended.\n'
          '- SUTURES: If you had resorbable sutures placed, then they will dissolve on its own\n'
          'in 7-10 days and you do not need to worry about suture removal.\n'
          '- SPECIAL CONSIDERATIONS – Trismus (stiffness) in the face muscles may cause\n'
          'difficulty in opening your mouth for a period of days. Moist heat compresses can\n'
          'minimize this condition. You may experience aching from other teeth. This\n'
          'discomfort is caused by referred pain and is a temporary condition. It is not unusual\n'
          'to develop bruising in the area of the extraction. There may be a slight elevation in\n'
          'temperature for 24-48 hours. If the fever persists, please contact our office.\n'
          '- FOLLOW-UP APPOINTMENTS: You may need to return to the office for a brief\n'
          'follow-up healing check.\n'
          'Following these instructions very closely will greatly help your comfort, and promote\n'
          'uneventful healing of the area. If any of the instructions are not followed, you might '
          'have\n'
          'significantly more discomfort, and the success of the procedure may be affected.'},
 {'document_id': 'D10',
  'title': 'Crown Lengthening Post-Operative Instructions',
  'doc_type': 'crown_lengthening',
  'effective_date': '2025-02-15',
  'is_current': True,
  'text': 'CROWN LENGTHENING\n'
          '- MOST IMPORTANT DIRECTION: You have been given an IRRIGATION SYRINGE and\n'
          'instructed how and when to use it. You may begin gentle rinsing with IRRIGATION\n'
          'SYRINGE using a saltwater solution (1 or 2 teaspoon salt + 8 ounces warm water).\n'
          'Be sure to RINSE AFTER EACH MEAL OR SNACK! If you keep your surgical site clean,\n'
          'you will heal, if you do not, it will cause secondary infection. Remember, it takes\n'
          'only one time for food to get into the surgical site and then secondary infection will\n'
          'start, so be sure to RINSE AFTER EACH MEAL OR SNACK!\n'
          '- CHLORHEXIDINE RINSING: In addition to warm salt water rinses after each meal\n'
          'or every 2-3 hours, you would also need to rinse with “liquid antibiotic” called\n'
          'Chlorhexidine. If you been given prescription at your appointment, then follow these\n'
          'instructions. After breakfast and dinner, or morning and evening, when you rinse\n'
          'with warm salt water and get the site cleaned, now you need to rinse with\n'
          'Chlorhexidine, use same syringe and do exactly the same step as with warm salt\n'
          'water, but now use Chlorhexidine.\n'
          '- DO NOT: Swish, suck through a straw, spit or smoke, it can all dislodge the clot.\n'
          'Keep anything sharp from entering the wound (crunchy food, toothpicks, eating\n'
          'utensils). Be sure to chew on the opposite side until the area heals\n'
          'completely. Understand that any food caught inside the extraction socket (place\n'
          'where tooth used to be) can cause infection, pain and additional surgery, so stay on\n'
          'soft or liquid food diet and give your body enough time to heal, BUT BE SURE TO\n'
          'RINSE AFTER EACH MEAL OR SNACK!\n'
          '- BLEEDING: When you leave the office, you might be biting on a gauze pad to control\n'
          'bleeding. Keep slight pressure on this gauze for at least 30 to 60 minutes. Don’t\n'
          'change it during this time; it needs to remain undisturbed while a clot forms in the\n'
          'extraction socket. After 30 minutes you may remove it. You may bite on another\n'
          'gauze or a tea bag for another 30 minutes if you feel it is still bleeding. Small\n'
          'amounts of blood in the saliva can make your saliva appear quite red. This is normal\n'
          'and may be noticed the rest of the day after the procedure.\n'
          '- DIET: Eat soft foods until the area is healed. Maintain a good, balanced diet, you\n'
          'need proper nutrition to heal. Drink plenty of water. Avoid alcohol for 48\n'
          'hours. Keeping the area clean from food debris will promote good healing.\n'
          '- SMOKING: Smoking should be stopped following surgery. Healing and success of\n'
          'the surgery will be substantially reduced by the cigarette smoke chemicals in your\n'
          'body. Also the suction created when inhaling cigarettes can dislodge the clot.\n'
          'Smokers are at greater risk of developing a painful condition called “Dry Socket”,\n'
          'which can hurt worse then a toothache and there is no cure for it. You must quit for\n'
          'at least 5 day to allow initial healing.\n'
          '- PAIN: Some discomfort is normal after surgery. To minimize pain, take Tylenol,\n'
          'Motrin or Advil, or similar non-aspirin pain reliever to maintain comfort. Start take '
          'it\n'
          'before the anesthesia wears off. If prescription pain medication is prescribed, take '
          'it\n'
          'as instructed on the label. Don’t exceed the dose on the label. Taking it with food\n'
          'will help reduce upset stomach. Avoid driving or operating heavy machinery when\n'
          'taking pain prescriptions. Do not drink alcohol while taking prescription pain\n'
          'medications. Strong pain medications, like Norco, must be only taken before\n'
          'bedtime.\n'
          '- NAUSEA: This is most often caused by taking pain medications on an empty\n'
          'stomach. Reduce nausea by preceding each pain pill with soft food, and taking the\n'
          'pill with a large glass of water.\n'
          '- SWELLING: Applying an ice bag to the face over the operated area will minimize\n'
          'swelling. Apply for 15 minutes, then remove for 15 minutes. Continue this for the\n'
          'first day.\n'
          '- NUMBNESS: The local anesthetic will cause you to be numb for several hours after\n'
          'you leave the office. Be very careful not to bite, chew, pinch, or scratch the numb\n'
          'area. Sometimes the extraction causes residual numbness or tingling for six weeks\n'
          'or longer. Contact our office if you experience these symptoms.\n'
          '- BRUSHING: Do not brush your teeth for the first 8 hours after surgery. After this, '
          'you\n'
          'must brush your teeth to reduce bacteria amount, but avoid the area of surgery for\n'
          'few days.\n'
          '- ACTIVITY: After leaving the office, rest and avoid strenuous activities for the\n'
          'remainder of the day. Keeping blood pressure lower will reduce bleeding and aid\n'
          'healing.\n'
          '- ANTIBIOTICS: If you were given an antibiotic prescription, take all of them as\n'
          'directed until they are gone. Women: some antibiotics can reduce the effectiveness\n'
          'of birth control pills. Use alternate birth control methods for two months.\n'
          '- SINUS: If your sinus was involved in the procedure, you should avoid blowing your\n'
          'nose or playing a wind musical instrument for one week. Use of decongestant\n'
          'medications might be recommended.\n'
          '- SUTURES: If you had resorbable sutures placed, then they will dissolve on its own\n'
          'in 7-10 days and you do not need to worry about suture removal. If you have sutures\n'
          'that do not resorb, then you need to return back to the office for a sutures removal.\n'
          '- SPECIAL CONSIDERATIONS – Trismus (stiffness) in the face muscles may cause\n'
          'difficulty in opening your mouth for a period of days. Moist heat compresses can\n'
          'minimize this condition. You may experience aching from other teeth. This\n'
          'discomfort is caused by referred pain and is a temporary condition. It is not unusual\n'
          'to develop bruising in the area of the extraction. There may be a slight elevation in\n'
          'temperature for 24-48 hours. If the fever persists, please contact our office.\n'
          '- FOLLOW-UP APPOINTMENTS: You WILL NEED TO RETURN to the office for a brief\n'
          'follow-up healing check. We want to make sure you get checks and evaluated and\n'
          'staying on proper healing course. You WILL NEED TO RETURN in 6-8 weeks to work\n'
          'on your definitive crown, remember that the crown you are wearing now is\n'
          'temporary only, if it does come off or break, you need to report to us so that we can\n'
          'get your crown back on to prevent for your tooth to decay and fail, so that you do not\n'
          'lose that tooth.\n'
          'Following these instructions very closely will greatly help your comfort, and promote\n'
          'uneventful healing of the area. If any of the instructions are not followed, you might '
          'have\n'
          'significantly more discomfort, and the success of the procedure may be affected.'},
 {'document_id': 'D11',
  'title': 'Dental Implant Placement Post-Operative Instructions',
  'doc_type': 'implant_placement',
  'effective_date': '2025-03-15',
  'is_current': True,
  'text': 'DENTAL IMPLANT PLACEMENT\n'
          '- DO NOT DISTURB THE AREA: For the next few days, and especially the first\n'
          '24 hours, it is very important to allow your body to form a good clot and start\n'
          'natural healing process.\n'
          '- DO NOT: Swish, suck through a straw, spit or smoke, it can all dislodge the\n'
          'clot. Keep anything sharp from entering the wound (crunchy food,\n'
          'toothpicks, eating utensils).\n'
          '- Be sure to chew on the opposite side until the area heals completely.\n'
          '- Understand that any food caught inside the extraction socket (place where\n'
          'tooth used to be) can cause infection, pain and additional surgery, so stay on\n'
          'soft or liquid food diet and give your body enough time to heal.\n'
          '- MOST IMPORTANT DIRECTION: You have been given an IRRIGATION\n'
          'SYRINGE and instructed how and when to use it. You may begin gentle\n'
          'rinsing with IRRIGATION SYRINGE using a saltwater solution (1 or 2 teaspoon\n'
          'salt + 8 ounces warm water). Be sure to RINSE AFTER EACH MEAL OR\n'
          'SNACK! If you keep your surgical site clean, you will heal, if you do not, it will\n'
          'cause secondary infection. Remember, it takes only one time for food to get\n'
          'into the surgical site and then secondary infection will start, so be sure to\n'
          'RINSE AFTER EACH MEAL OR SNACK!\n'
          '- BLEEDING: When you leave the office, you might be biting on a gauze pad to\n'
          'control bleeding. Keep slight pressure on this gauze for at least 30 to 60\n'
          'minutes. Don’t change it during this time; it needs to remain undisturbed\n'
          'while a clot forms in the extraction socket. After 30 minutes you may remove\n'
          'it. You may bite on another gauze or a tea bag for another 30 minutes if you\n'
          'feel it is still bleeding. Small amounts of blood in the saliva can make your\n'
          'saliva appear quite red. This is normal and may be noticed the rest of the day\n'
          'after the procedure.\n'
          '- DIET: Eat soft foods until the area is healed. Maintain a good, balanced diet,\n'
          'you need proper nutrition to heal. Drink plenty of water. Avoid alcohol for 48\n'
          'hours. Keeping the area clean from food debris will promote good healing.\n'
          '- SMOKING: Smoking should be stopped following surgery. Healing and\n'
          'success of the surgery will be substantially reduced by the cigarette smoke\n'
          'chemicals in your body. Also the suction created when inhaling cigarettes\n'
          'can dislodge the clot. Smokers are at greater risk of developing a painful\n'
          'condition called “Dry Socket”, which can hurt worse then a toothache and\n'
          'there is no cure for it. You must quit for at least 5 day to allow initial healing.\n'
          '- PAIN: Some discomfort is normal after surgery. To minimize pain, take\n'
          'Tylenol, Motrin or Advil, or similar non-aspirin pain reliever to maintain\n'
          'comfort. Start take it before the anesthesia wears off. If prescription pain\n'
          'medication is prescribed, take it as instructed on the label. Don’t exceed the\n'
          'dose on the label. Taking with food will help reduce upset stomach. Avoid\n'
          'driving or operating heavy machinery when taking pain prescriptions. Do not\n'
          'drink alcohol while taking prescription pain medications. Strong pain\n'
          'medications, like Norco, must be only taken before bedtime.\n'
          '- NAUSEA: This is most often caused by taking pain medications on an empty\n'
          'stomach. Reduce nausea by preceding each pain pill with soft food, and\n'
          'taking the pill with a large glass of water.\n'
          '- SWELLING: Applying an ice bag to the face over the operated area will\n'
          'minimize swelling. Apply for 15 minutes, then remove for 15 minutes.\n'
          'Continue this for the first day.\n'
          '- NUMBNESS: The local anesthetic will cause you to be numb for several\n'
          'hours after you leave the office. Be very careful not to bite, chew, pinch, or\n'
          'scratch the numb area. Sometimes the extraction causes residual numbness\n'
          'or tingling for six weeks or longer. Contact our office if you experience these\n'
          'symptoms.\n'
          '- BRUSHING: Do not brush your teeth for the first 8 hours after surgery. After\n'
          'this, you must brush your teeth to reduce bacteria amount, but avoid the\n'
          'area of surgery for few days.\n'
          '- RINSING: Avoid all rinsing or swishing for 24 hours after extraction. Rinsing\n'
          'can disturb the formation of a healing blood clot which is essential to proper\n'
          'healing. This could cause bleeding and risk of dry socket. After 24 hours you\n'
          'may begin gentle rinsing with a saltwater solution (1 or 2 teaspoon salt + 8\n'
          'ounces warm water) after each meal or snack. Avoid commercial mouth\n'
          'rinses that contain alcohol.\n'
          '- ACTIVITY: After leaving the office, rest and avoid strenuous activities for the\n'
          'remainder of the day. Keeping blood pressure lower will reduce bleeding and\n'
          'aid healing.\n'
          '- ANTIBIOTICS: If you were given an antibiotic prescription, take all of them as\n'
          'directed until they are gone. Women: some antibiotics can reduce the\n'
          'effectiveness of birth control pills. Use alternate birth control methods for\n'
          'two months.\n'
          '- SINUS: If your sinus was involved in the procedure, you should avoid blowing\n'
          'your nose or playing a wind musical instrument for one week. Use of\n'
          'decongestant medications might be recommended.\n'
          '- SUTURES: If you had resorbable sutures placed, then they will dissolve on\n'
          'its own in 7-10 days and you do not need to worry about suture removal.\n'
          '- SPECIAL CONSIDERATIONS – Trismus (stiffness) in the face muscles may\n'
          'cause difficulty in opening your mouth for a period of days. Moist heat\n'
          'compresses can minimize this condition. You may experience aching from\n'
          'other teeth. This discomfort is caused by referred pain and is a temporary\n'
          'condition. It is not unusual to develop bruising in the area of the extraction.\n'
          'There may be a slight elevation in temperature for 24-48 hours. If the fever\n'
          'persists, please contact our office.\n'
          '- FOLLOW-UP APPOINTMENTS: You may need to return to the office for a\n'
          'brief follow-up healing check.\n'
          'Following these instructions very closely will greatly help your comfort, and promote\n'
          'uneventful healing of the area. If any of the instructions are not followed, you might '
          'have\n'
          'significantly more discomfort, and the success of the procedure may be affected.'},
 {'document_id': 'D12',
  'title': 'Deep Cleaning (Scaling and Root Planing) Post-Treatment Instructions',
  'doc_type': 'periodontal_scaling',
  'effective_date': '2025-06-01',
  'is_current': True,
  'text': 'DEEP CLEANING (Scaling and Root Planing)\n'
          'To minimize the discomfort and encourage proper healing following your scaling and '
          'root\n'
          'planing, follow these instructions:\n'
          '- After the procedure, take aspirin, acetaminophen (Tylenol®), or ibuprofen (Advil®)\n'
          'before the anesthetic wears off. Continue to take over the counter pain medication\n'
          'for the next two days even if you do not have any discomfort.\n'
          '- A saltwater solution (1 OR 2 teaspoon salt + 8 ounces warm water) swished in your\n'
          'mouth for 2 to 3 minutes every hour may make your mouth more comfortable.\n'
          '- Use a soft toothbrush and floss at least two times a day. Be gentle and clean\n'
          'thoroughly. Plaque (Bacteria) will start to grow and removing it with brushing and\n'
          'flossing at least twice a day will promote a positive outcome of the treatment.\n'
          '- Slight bleeding may occur while brushing as the tissues begin to heal and is normal\n'
          'up to 4 weeks following treatment.\n'
          '- Avoid strong spicy seasonings, and hard crunchy foods for the next few days.\n'
          '- Smoking should be stopped. Success of the treatment will be substantially reduced\n'
          'by the cigarette smoke chemicals in your body.\n'
          '- As the tissues heal, some temporary sensitivity to cold may occur. Use a\n'
          'desensitizing toothpaste (such as Sensodyne® without whitening), or fluoride gel\n'
          '(such as Prevident® or Gel-Kam®) frequently (at least 4 times/day) for 1 to 2 weeks.\n'
          'Also, the cleaner the teeth are kept, the less sensitive they will be.\n'
          '- Faithfully use any other oral hygiene aids that have been recommended (floss,\n'
          'waterpick, chlorhexidine, rubber tip, Sonicare®, Proxabrush®, Peridex® mouthrinse,\n'
          'etc).\n'
          '- If you had antibiotics like Arestin or Atridox applied at the time of deep cleaning, '
          'do\n'
          'all the recommended post op instructions but DO NOT floss or waterpick the\n'
          'treated teeth with antibiotics for at least 10 days.\n'
          '- Always Follow Up. Because the bacteria that cause periodontal disease are\n'
          'persistent, the infection can return. Please be sure to make follow-\n'
          'up appointments with your dental professional to maintain healthy gums and\n'
          'teeth.\n'
          '- Keep Your Scheduled Appointments. It is important to keep all of your dental\n'
          'appointments so that your dental professional can re-examine your gums,\n'
          'make sure the infection is under control, and measure the success of your\n'
          'treatment.'},
 {'document_id': 'D13',
  'title': 'Gum Graft Post-Operative Instructions',
  'doc_type': 'gum_graft',
  'effective_date': '2025-04-20',
  'is_current': True,
  'text': 'GUM GRAFT\n'
          '- MOST IMPORTANT DIRECTION: You have been given an IRRIGATION\n'
          'SYRINGE and instructed how and when to use it. You may begin gentle\n'
          'rinsing with IRRIGATION SYRINGE using a saltwater solution (1 or 2\n'
          'teaspoon salt + 8 ounces warm water). Be sure to RINSE AFTER EACH\n'
          'MEAL OR SNACK! If you keep your surgical site clean, you will heal, if\n'
          'you do not, it will cause secondary infection. Remember, it takes only\n'
          'one time for food to get into the surgical site and then secondary\n'
          'infection will start, so be sure to RINSE AFTER EACH MEAL OR\n'
          'SNACK!\n'
          '- CHLORHEXIDINE RINSING: In addition to warm salt water rinses\n'
          'after each meal or every 2-3 hours, you would also need to rinse with\n'
          '“liquid antibiotic” called Chlorhexidine. If you been given prescription\n'
          'at your appointment, then follow these instructions. After breakfast\n'
          'and dinner, or morning and evening, when you rinse with warm salt\n'
          'water and get the site cleaned, now you need to rinse with\n'
          'Chlorhexidine, use same syringe and do exactly the same step as with\n'
          'warm salt water, but now use Chlorhexidine.\n'
          '- STENT/MATRIX: If you being given a clear matrix/stent for your upper jaw to\n'
          'cover the donor site on the roof of your mouth, you may wear it for all the\n'
          'time or during the day, depends on how much discomfort you feel. Take the\n'
          'stent out after each time you each and rinse it under luke warm water, use\n'
          'toothbrush and soap to clean it, do not use toothpaste. Everytime you rinse,\n'
          'take the matrix out and make sure the solution gets to the wound. Do not put\n'
          'matrix into the hot water.\n'
          '- DO NOT: Swish, suck through a straw, spit or smoke, it can all dislodge the\n'
          'clot. Keep anything sharp from entering the wound (crunchy food,\n'
          'toothpicks, eating utensils). Be sure to chew on the opposite side until the\n'
          'area heals completely. Understand that any food caught inside the\n'
          'extraction socket (place where tooth used to be) can cause infection, pain\n'
          'and additional surgery, so stay on soft or liquid food diet and give your body\n'
          'enough time to heal, BUT BE SURE TO RINSE AFTER EACH MEAL OR SNACK!\n'
          '- BLEEDING: When you leave the office, you might be biting on a gauze pad to\n'
          'control bleeding. Keep slight pressure on this gauze for at least 30 to 60\n'
          'minutes. Don’t change it during this time; it needs to remain undisturbed\n'
          'while a clot forms in the extraction socket. After 30 minutes you may remove\n'
          'it. You may bite on another gauze or a tea bag for another 30 minutes if you\n'
          'feel it is still bleeding. Small amounts of blood in the saliva can make your\n'
          'saliva appear quite red. This is normal and may be noticed the rest of the day\n'
          'after the procedure.\n'
          '- DIET: Eat soft foods until the area is healed. Maintain a good, balanced diet,\n'
          'you need proper nutrition to heal. Drink plenty of water. Avoid alcohol for 48\n'
          'hours. Keeping the area clean from food debris will promote good healing.\n'
          '- SMOKING: Smoking should be stopped following surgery. Healing and\n'
          'success of the surgery will be substantially reduced by the cigarette smoke\n'
          'chemicals in your body. Also the suction created when inhaling cigarettes\n'
          'can dislodge the clot. Smokers are at greater risk of developing a painful\n'
          'condition called “Dry Socket”, which can hurt worse then a toothache and\n'
          'there is no cure for it. You must quit for at least 5 day to allow initial healing.\n'
          '- PAIN: Some discomfort is normal after surgery. To minimize pain, take\n'
          'Tylenol, Motrin or Advil, or similar non-aspirin pain reliever to maintain\n'
          'comfort. Start take it before the anesthesia wears off. If prescription pain\n'
          'medication is prescribed, take it as instructed on the label. Don’t exceed the\n'
          'dose on the label. Taking it with food will help reduce upset stomach. Avoid\n'
          'driving or operating heavy machinery when taking pain prescriptions. Do not\n'
          'drink alcohol while taking prescription pain medications. Strong pain\n'
          'medications, like Norco, must be only taken before bedtime.\n'
          '- NAUSEA: This is most often caused by taking pain medications on an empty\n'
          'stomach. Reduce nausea by preceding each pain pill with soft food, and\n'
          'taking the pill with a large glass of water.\n'
          '- SWELLING: Applying an ice bag to the face over the operated area will\n'
          'minimize swelling. Apply for 15 minutes, then remove for 15 minutes.\n'
          'Continue this for the first day.\n'
          '- NUMBNESS: The local anesthetic will cause you to be numb for several\n'
          'hours after you leave the office. Be very careful not to bite, chew, pinch, or\n'
          'scratch the numb area. Sometimes the extraction causes residual numbness\n'
          'or tingling for six weeks or longer. Contact our office if you experience these\n'
          'symptoms.\n'
          '- BRUSHING: Do not brush your teeth for the first 8 hours after surgery. After\n'
          'this, you must brush your teeth to reduce bacteria amount, but avoid the\n'
          'area of surgery for few days.\n'
          '- ACTIVITY: After leaving the office, rest and avoid strenuous activities for the\n'
          'remainder of the day. Keeping blood pressure lower will reduce bleeding and\n'
          'aid healing.\n'
          '- ANTIBIOTICS: If you were given an antibiotic prescription, take all of them as\n'
          'directed until they are gone. Women: some antibiotics can reduce the\n'
          'effectiveness of birth control pills. Use alternate birth control methods for\n'
          'two months.\n'
          '- SINUS: If your sinus was involved in the procedure, you should avoid blowing\n'
          'your nose or playing a wind musical instrument for one week. Use of\n'
          'decongestant medications might be recommended.\n'
          '- SUTURES: If you had resorbable sutures placed, then they will dissolve on\n'
          'its own in 7-10 days and you do not need to worry about suture removal. If\n'
          'you have sutures that do not resorb, then you need to return back to the\n'
          'office for a sutures removal.\n'
          '- SPECIAL CONSIDERATIONS – Trismus (stiffness) in the face muscles may\n'
          'cause difficulty in opening your mouth for a period of days. Moist heat\n'
          'compresses can minimize this condition. You may experience aching from\n'
          'other teeth. This discomfort is caused by referred pain and is a temporary\n'
          'condition. It is not unusual to develop bruising in the area of the extraction.\n'
          'There may be a slight elevation in temperature for 24-48 hours. If the fever\n'
          'persists, please contact our office.\n'
          '- FOLLOW-UP APPOINTMENTS: You may need to return to the office for a\n'
          'brief follow-up healing check.\n'
          'Following these instructions very closely will greatly help your comfort, and promote\n'
          'uneventful healing of the area. If any of the instructions are not followed, you might '
          'have\n'
          'significantly more discomfort, and the success of the procedure may be affected.'},
 {'document_id': 'D14',
  'title': 'Invisalign Clear Aligner Care Instructions',
  'doc_type': 'clear_aligner_care',
  'effective_date': '2025-05-15',
  'is_current': True,
  'text': 'INVISALIGN BRACES CARE\n'
          '- Taking care of Invisalign clear aligners is very simple. Every time you brush your\n'
          'teeth you can brush your aligners but without a toothpaste. The best way to clean\n'
          'your aligners is to use the Invisalign cleaning kit. They can also be soaked and\n'
          'cleaned in a denture cleaner.\n'
          '- It is not advisable to remove your aligners at night.\n'
          '- Thanks to the removable nature though, you can eat and drink whatever you want\n'
          'while in treatment. In fact, you’re required to remove your aligners to eat and drink.\n'
          'So, unlike undergoing traditional treatment using wires and brackets, there is no\n'
          'need to restrict your consumption of any of your favorite foods and snacks unless\n'
          'instructed otherwise by Dr. Denisovich.\n'
          '- Also, it is important that you brush your teeth after each meal and prior to re-\n'
          'inserting your aligners to maintain fresh breath and proper hygiene.\n'
          '- Dr. Denisovich discourages smoking while wearing aligners because it is possible\n'
          'for the aligners to become discolored.\n'
          '- You cannot chew gum while wearing your aligners. It will stick to the aligners.\n'
          '- We recommend removing your aligners for all meals and snacks.\n'
          'Trays May Cause Some Irritation To The Lips, Tongue, And Cheeks\n'
          'If This Occurs\n'
          '- Use wax to stop the irritation\n'
          '- Rinse with warm salt water to help irritation heal\n'
          '- Products like Zilactin-B or a peroxide based rinse can be helpful, and are available\n'
          'over the counter\n'
          'Things TO DO During Invisalign Treatment:\n'
          '- Wear each set of trays for at least 2 weeks, unless otherwise instructed\n'
          '- Wear each set of trays for 18- 20 hours a day, unless otherwise instructed\n'
          '- Use denture cleaner tablets to clean trays at least once a day\n'
          '- Always place trays in case when not in your mouth\n'
          '- Always remove trays by starting from the back molars\n'
          '- Always place trays in from the front of your mouth first, and then move to the back\n'
          'teeth\n'
          '- You may drink water while wearing trays or use a straw for dark liquids\n'
          '- If possible, brush or rinse before placing trays back in your mouth\n'
          '- Clenching into trays w/ your aligner chewies during the first 2-3 days helps teeth\n'
          'move faster and relieve pressure. Only clench for 30-40 sec each quadrant, and\n'
          'repeat for 5-10 min, but only do this if you have no history of jaw problems\n'
          '- Wear trays as instructed in sequential order\n'
          '- Keep 2 to 3 of your previous trays in a clean plastic bag\n'
          'Things NOT TO DO During Invisalign Treatment:\n'
          '- Throw away trays\n'
          '- Leave trays out of mouth for long periods of time\n'
          '- Chew gum with aligners in your mouth\n'
          '- Leave trays in hot vehicle, or boil them (They are plastic!)\n'
          '- Leave trays sitting out for pets or small children to chew on\n'
          '- Wrap trays in a napkin (You will throw them away acidently!)\n'
          '- Place trays in your pocket without a case\n'
          '- Have dental work done while in treatment, EXCEPT for regular checkups and\n'
          'cleanings\n'
          '- Eat while wearing trays\n'
          '- Remove trays from the front teeth first\n'
          '- Drink dark teas, coffee or soda with trays in (Use a straw)\n'
          '- Set trays on table at a restaurant\n'
          '- Bite trays into position, this may damage them\n'
          '- Use mouthwash or toothpaste on trays\n'
          'Teeth may begin to feel slightly mobile or loose during treatment, but this is normal.\n'
          'Non-compliant wear of aligners may result in the need for new impressions and '
          'additional\n'
          'fees'},
 {'document_id': 'D15',
  'title': 'Post-Operative Care After a Dental Night Guard',
  'doc_type': 'night_guard_care',
  'effective_date': '2025-06-15',
  'is_current': True,
  'text': 'Post-Operative Care After a Dental Night Guard\n'
          'When traveling, store your Night Guard in a clean, dry place. A hard case is ideal for\n'
          'protecting your night guard from damage and dust. You can simply use dental pod '
          'tablets\n'
          'and dissolve them in a cup of water to soak your night guard during the day when the '
          'actual\n'
          'dental pod ultrasonic cleaner is not available.\n'
          'Bring your night guard to all dental appointments. We will need to check it for wear '
          'and tear\n'
          'and make sure it is still fitting properly.\n'
          'Wearing Your Night Guard:\n'
          'Wear your night guard as directed by your dentist. This is usually every night while '
          'you\n'
          'sleep. REMEMBER - It only works when you wear it.\n'
          'Do not chew on your night guard. This can cause it to break or become misshapen.\n'
          'Do not let your lovely dog or a cat find it, if they do, then this would be the last '
          'time you will\n'
          'see it.\n'
          'If your night guard feels uncomfortable, contact us. You may need adjustments for a '
          'better\n'
          'fit.\n'
          'Additional Tips:\n'
          'You may experience some minor discomfort when you first start wearing your night '
          'guard.\n'
          'This is normal and should subside within a few weeks.\n'
          'You may also experience increased saliva production while wearing your night guard. '
          'This\n'
          'is also normal and should subside over time.\n'
          'By following these post-operative care instructions, you can help ensure that your '
          'night\n'
          'guard is effective and lasts for many years.\n'
          'Please note: These are general post-operative care instructions. It is important to '
          'follow the\n'
          'specific instructions provided to you at your visit.\n'
          'Over time, night guards can wear down and lose their effectiveness.'},
 {'document_id': 'D16',
  'title': 'Partial or Complete Denture Care Instructions',
  'doc_type': 'denture_care',
  'effective_date': '2025-07-01',
  'is_current': True,
  'text': 'PARTIAL OR COMPLETE DENTURE\n'
          '- New dentures always require a period of adjustment. First-time denture patients\n'
          'may require several weeks or months to get used to their new appliance. Speech\n'
          'may be altered, and may require adaptation of the tongue and lips.\n'
          '- For the first few days, you should wear your dentures for as long as possible, and\n'
          'chew soft food in small bites on both sides of your mouth. Remember, dentures do\n'
          'not have the same chewing efficiency as natural teeth and may affect your taste of\n'
          'food.\n'
          '- If your bite feels uneven after several days, please let us know, we can adjust the\n'
          'way your teeth contact at follow-up visits.\n'
          '- It is not unusual for sore spots to develop in isolated areas of the mouth. These\n'
          'areas can be relieved easily at follow-up appointments.\n'
          '- If a severe sore spot develops which prevents wearing the denture and an\n'
          'appointment is made for adjustment, please wear the denture for 24 hours prior to\n'
          'the appointment. This will greatly aid in locating the exact area, and make\n'
          'adjustments significantly easier and more predictable.\n'
          '- Proper cleaning of your denture is important to prevent stains and bacteria from\n'
          'accumulating on your appliance. Since cleaning procedures differ for various types\n'
          'of appliances, please follow the directions given to you at your insertion\n'
          'appointment.\n'
          '- DO NOT wear your complete or partial dentures to bed. It is important to allow your\n'
          'gum tissues and jaw bones to rest in order to prevent further tissue irritation,\n'
          'infection, and future bone shrinkage.\n'
          '- Over time, or with weight change, the supporting gum tissues and bone will change\n'
          'shape and size. Periodic relines of your dentures may be necessary to ensure a\n'
          'retentive fit. Denture teeth will wear or chip over time. For this reason, an annual\n'
          'check of your tissues and dentures is recommended.'},
 {'document_id': 'D17',
  'title': 'Tooth Extraction - General Dental Surgery Post-Operative Instructions',
  'doc_type': 'tooth_extraction_general',
  'effective_date': '2025-01-25',
  'is_current': True,
  'text': 'TOOTH EXTRACTION - GENERAL DENTAL SURGERY\n'
          '- MOST IMPORTANT DIRECTION: You have been given an IRRIGATION\n'
          'SYRINGE and instructed how and when to use it. You may begin gentle\n'
          'rinsing with IRRIGATION SYRINGE using a saltwater solution (1 or 2 teaspoon\n'
          'salt + 8 ounces warm water). Be sure to RINSE AFTER EACH MEAL OR\n'
          'SNACK! If you keep your surgical site clean, you will heal, if you do not, it will\n'
          'cause secondary infection. Remember, it takes only one time for food to get\n'
          'into the surgical site and then secondary infection will start, so be sure to\n'
          'RINSE AFTER EACH MEAL OR SNACK!\n'
          '- DO NOT DISTURB THE AREA: For the next few days, and especially the first\n'
          '24 hours, it is very important to allow your body to form a good clot and start\n'
          'natural healing process.\n'
          '- DO NOT: Swish, suck through a straw, spit or smoke, it can all dislodge the\n'
          'clot. Keep anything sharp from entering the wound (crunchy food,\n'
          'toothpicks, eating utensils). Be sure to chew on the opposite side until the\n'
          'area heals completely. Understand that any food caught inside the\n'
          'extraction socket (place where tooth used to be) can cause infection, pain\n'
          'and additional surgery, so stay on soft or liquid food diet and give your body\n'
          'enough time to heal, BUT BE SURE TO RINSE AFTER EACH MEAL OR SNACK!\n'
          '- BLEEDING: When you leave the office, you might be biting on a gauze pad to\n'
          'control bleeding. Keep slight pressure on this gauze for at least 30 to 60\n'
          'minutes. Don’t change it during this time; it needs to remain undisturbed\n'
          'while a clot forms in the extraction socket. After 30 minutes you may remove\n'
          'it. You may bite on another gauze or a tea bag for another 30 minutes if you\n'
          'feel it is still bleeding. Small amounts of blood in the saliva can make your\n'
          'saliva appear quite red. This is normal and may be noticed the rest of the day\n'
          'after the procedure.\n'
          '- DIET: Eat soft foods until the area is healed. Maintain a good, balanced diet,\n'
          'you need proper nutrition to heal. Drink plenty of water. Avoid alcohol for 48\n'
          'hours. Keeping the area clean from food debris will promote good healing.\n'
          '- SMOKING: Smoking should be stopped following surgery. Healing and\n'
          'success of the surgery will be substantially reduced by the cigarette smoke\n'
          'chemicals in your body. Also the suction created when inhaling cigarettes\n'
          'can dislodge the clot. Smokers are at greater risk of developing a painful\n'
          'condition called “Dry Socket”, which can hurt worse then a toothache and\n'
          'there is no cure for it. You must quit for at least 5 day to allow initial healing.\n'
          '- PAIN: Some discomfort is normal after surgery. To minimize pain, take\n'
          'Tylenol, Motrin or Advil, or similar non-aspirin pain reliever to maintain\n'
          'comfort. Start take it before the anesthesia wears off. If prescription pain\n'
          'medication is prescribed, take it as instructed on the label. Don’t exceed the\n'
          'dose on the label. Taking with food will help reduce upset stomach. Avoid\n'
          'driving or operating heavy machinery when taking pain prescriptions. Do not\n'
          'drink alcohol while taking prescription pain medications. Strong pain\n'
          'medications, like Norco, must be only taken before bedtime.\n'
          '- NAUSEA: This is most often caused by taking pain medications on an empty\n'
          'stomach. Reduce nausea by preceding each pain pill with soft food, and\n'
          'taking the pill with a large glass of water.\n'
          '- SWELLING: Applying an ice bag to the face over the operated area will\n'
          'minimize swelling. Apply for 15 minutes, then remove for 15 minutes.\n'
          'Continue this for the first day.\n'
          '- NUMBNESS: The local anesthetic will cause you to be numb for several\n'
          'hours after you leave the office. Be very careful not to bite, chew, pinch, or\n'
          'scratch the numb area. Sometimes the extraction causes residual numbness\n'
          'or tingling for six weeks or longer. Contact our office if you experience these\n'
          'symptoms.\n'
          '- BRUSHING: Do not brush your teeth for the first 8 hours after surgery. After\n'
          'this, you must brush your teeth to reduce bacteria amount, but avoid the\n'
          'area of surgery for few days.\n'
          '- RINSING: You may begin gentle rinsing with IRRIGATION SYRINGE using a\n'
          'saltwater solution (1 or 2 teaspoon salt + 8 ounces warm water). Be sure to\n'
          'RINSE AFTER EACH MEAL OR SNACK! If you keep your surgical site clean,\n'
          'you will heal, if you do not, it will cause secondary infection. Remember, it\n'
          'takes only one time for food to get into the surgical site and then secondary\n'
          'infection will start, so be sure to RINSE AFTER EACH MEAL OR SNACK! Avoid\n'
          'commercial mouth rinses that contain alcohol.\n'
          '- ACTIVITY: After leaving the office, rest and avoid strenuous activities for the\n'
          'remainder of the day. Keeping blood pressure lower will reduce bleeding and\n'
          'aid healing.\n'
          '- ANTIBIOTICS: If you were given an antibiotic prescription, take all of them as\n'
          'directed until they are gone. Women: some antibiotics can reduce the\n'
          'effectiveness of birth control pills. Use alternate birth control methods for\n'
          'two months.\n'
          '- SINUS: If your sinus was involved in the procedure, you should avoid blowing\n'
          'your nose or playing a wind musical instrument for one week. Use of\n'
          'decongestant medications might be recommended.\n'
          '- SUTURES: If you had resorbable sutures placed, then they will dissolve on\n'
          'its own in 7-10 days and you do not need to worry about suture removal. If\n'
          'you have sutures that do not resorb, then you need to return back to the\n'
          'office for a sutures removal.\n'
          '- SPECIAL CONSIDERATIONS – Trismus (stiffness) in the face muscles may\n'
          'cause difficulty in opening your mouth for a period of days. Moist heat\n'
          'compresses can minimize this condition. You may experience aching from\n'
          'other teeth. This discomfort is caused by referred pain and is a temporary\n'
          'condition. It is not unusual to develop bruising in the area of the extraction.\n'
          'There may be a slight elevation in temperature for 24-48 hours. If the fever\n'
          'persists, please contact our office.\n'
          '- FOLLOW-UP APPOINTMENTS: You may need to return to the office for a\n'
          'brief follow-up healing check.\n'
          'Following these instructions very closely will greatly help your comfort, and promote\n'
          'uneventful healing of the area. If any of the instructions are not followed, you might '
          'have\n'
          'significantly more discomfort, and the success of the procedure may be affected.'},
 {'document_id': 'D18',
  'title': 'White (Composite) Fillings Post-Operative Instructions',
  'doc_type': 'composite_fillings',
  'effective_date': '2025-02-25',
  'is_current': True,
  'text': 'WHITE FILLINGS\n'
          '- When anesthesia has been used, your lips, teeth, and tongue may be numb for\n'
          'several hours after the appointment. Avoid any chewing until the numbness has\n'
          'completely worn off. It is easy to bite or burn your tongue or lip while numb.\n'
          '- It is normal to experience some hot, cold and pressure sensitivity after your\n'
          'appointment. Your gums may be sore for several days.\n'
          '- Rinse your mouth three times a day with warm salt water (put a teaspoon of salt in a\n'
          'cup of warm water, rinse and spit) to reduce pain and swelling.\n'
          '- Your new composite fillings are fully hardened before you even leave the office and\n'
          'you can chew shortly after the numbing goes away!\n'
          '- One of the most common problems following filling placement with anesthesia, is\n'
          'an uneven bite. It is simply because its hard for the patient to feel correct bite due\n'
          'to numbness. Most of the time it resolves within few days by it self.\n'
          '- When decay is deep, a medication is placed under the filling to help preserve nerve\n'
          'vitality to avoid root canal. This procedure has very good success rate, but\n'
          'occasionally root canal might still be required.'},
 {'document_id': 'D19',
  'title': 'Take-Home Whitening Trays - General Instructions',
  'doc_type': 'whitening_take_home_general',
  'effective_date': '2025-03-25',
  'is_current': True,
  'text': 'WHITENING - TAKE HOME TRAYS\n'
          'While bleaching or whitening your teeth, normal oral hygiene measures should be '
          'followed\n'
          '(flossing and brushing).\n'
          '- Bead the bleaching gel inside your custom tray, using caution not to overload the\n'
          'tray. Insert the tray into your mouth over teeth and gently wipe any excess gel from\n'
          'around the edges of your tray, taking extra care not to leave any gel on your gums.\n'
          'Irritated gums usually means you have used too much gel in your tray.\n'
          '- Wear the tray for brief time, 30 minutes, to see how your mouth reacts to it. If you\n'
          'feel comfortable, then wear the trays as directed to you at your delivery\n'
          'appointment. The time of wear is based on the concentration and type of bleaching\n'
          'material utilized for your case. If the sensitivity is too uncomfortable, stop and\n'
          'reduce time and/or frequency. Usually sensitivity stops after several days.\n'
          '- While bleaching, do not rinse your mouth, since this may dilute the bleaching agent.\n'
          '- Do not swallow the whitening gel.\n'
          '- Never drink, eat, or smoke while wearing your tray.\n'
          '- Do not consume any red, black or staining foods like: coffee, sodas, red wine, etc.\n'
          '- After completing your bleaching session, remove the tray and gently clean it with a\n'
          'toothbrush and tap water without toothpaste. Rinse it completely, and let dry\n'
          'thoroughly before the next session.\n'
          'If you develop severe sensitivity or pain stop bleaching and please contact our office '
          'at\n'
          'your convenience.'}]

docs_df = pd.DataFrame(DOCUMENTS)
docs_df[['document_id', 'title', 'doc_type', 'effective_date', 'is_current']]

,document_id,title,doc_type,effective_date,is_current
0,D1,Post-Operative Instructions for Crown or Bridges,crown_bridge_care,2021-04-01,False
1,D2,Root Planing and Scaling for Gum Disease,periodontal_scaling,2020-09-01,False
2,D3,Post-Operative Instructions Following Oral Surgery,oral_surgery_general,2025-01-10,True
3,D4,Post-op Instructions for Implants and Extractions,implant_extraction_general,2025-02-05,True
4,D5,Post-Operative Instructions for Root Canal Treatment,root_canal,2025-03-01,True
5,D6,Opalescence Take-Home Whitening Step-by-Step Instructions,whitening_take_home_opalescence,2025-04-01,True
6,D7,Post-Operative Instructions After Bone Graft Surgery,bone_graft,2025-01-20,True
7,D8,Brushing With a Disclosing Agent (Home Hygiene Guide),home_hygiene_technique,2025-05-01,True
8,D9,Crown or Bridge Post-Operative Care (Updated),crown_bridge_care,2025-08-01,True
9,D10,Crown Lengthening Post-Operative Instructions,crown_lengthening,2025-02-15,True


In [20]:
print(f"Total documents: {len(docs_df)}")
print(f"Outdated documents: {(~docs_df['is_current']).sum()}  ->  {docs_df.loc[~docs_df['is_current'], 'document_id'].tolist()}")
print(f"Doc types shared by an outdated + current pair:")
shared = docs_df.groupby('doc_type')['document_id'].apply(list)
for dtype, ids in shared.items():
    if len(ids) > 1:
        print(f"  {dtype}: {ids}")

Total documents: 19
Outdated documents: 2  ->  ['D1', 'D2']
Doc types shared by an outdated + current pair:
  crown_bridge_care: ['D1', 'D9']
  periodontal_scaling: ['D2', 'D12']


## Section 1b — Text Preprocessing Profile (connecting Lab 5)

Lab 5 taught that there is **no universal preprocessing pipeline** — the right amount of cleaning
depends on the task. For this RAG pipeline:

- **TF-IDF / BM25 (lexical)** benefit from a *light* cleaning pass (lowercase + whitespace
  normalization) so surface variation like `"Trismus"` vs `"trismus"` doesn't create separate
  vocabulary entries, but we deliberately **do not** stem/lemmatize or strip stopwords/numbers,
  because dosages, hour counts (`"24 hours"`, `"48hours"`) and drug names are exactly the kind of
  information Lab 5 warned not to delete.
- **Semantic embeddings** are fed the *raw* chunk text, because sentence-transformer models are
  trained on natural, unprocessed sentences — aggressive cleaning would only throw away context
  the model already knows how to use.

This reuses the `preprocess_text()` helper built in Lab 5, with a `minimal_clean` profile.

In [21]:
def remove_urls(text):
    return re.sub(r"http\S+|www\.\S+", "", text)

def normalize_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

def preprocess_text(text, lowercase=True, remove_url=True, normalize_space=True):
    # Light-clean profile from Lab 5 (minimal_clean): safe for lexical retrieval,
    # keeps numbers, negation words and punctuation-adjacent meaning intact.
    if lowercase:
        text = text.lower()
    if remove_url:
        text = remove_urls(text)
    if normalize_space:
        text = normalize_whitespace(text)
    return text

# quick before/after demo on one real chunk of collected data
sample_raw = DOCUMENTS[6]['text'][:180]
print("RAW  :", sample_raw)
print("CLEAN:", preprocess_text(sample_raw))

RAW  : BONE GRAFT
- MOST IMPORTANT DIRECTION: You have been given an IRRIGATION SYRINGE and
instructed how and when to use it. You may begin gentle rinsing with IRRIGATION
SYRINGE using a
CLEAN: bone graft - most important direction: you have been given an irrigation syringe and instructed how and when to use it. you may begin gentle rinsing with irrigation syringe using a


## Section 2 — Text Chunking

Long documents are split into overlapping **chunks of 38 words with a 10-word overlap** (step =
28 words), exactly the fixed overlapping word-window chunking style taught in Lab 8 and reused
unchanged in Lab 9 ("Single-Path Chunking"), so that:
- No single chunk exceeds the retriever/LLM's useful context size.
- Overlap prevents cutting an instruction in half between two chunks.

Each chunk keeps its parent document's full metadata (including `is_current`, needed for context
filtering later), and a `search_text` field prepends the title and `doc_type` to the chunk text
so lexical/semantic retrievers get extra context clues.

In [22]:
def chunk_text(text, chunk_size=38, overlap=10):
    words = text.split()
    step = chunk_size - overlap
    chunks, i = [], 0
    while i < len(words):
        chunks.append(" ".join(words[i:i + chunk_size]))
        if i + chunk_size >= len(words):
            break
        i += step
    return chunks

rows = []
for doc in DOCUMENTS:
    cleaned_text = preprocess_text(doc['text'])
    for idx, ch in enumerate(chunk_text(doc['text'])):
        rows.append({
            'chunk_id': f"{doc['document_id']}_{idx}",
            'document_id': doc['document_id'],
            'title': doc['title'],
            'doc_type': doc['doc_type'],
            'effective_date': doc['effective_date'],
            'is_current': doc['is_current'],
            'chunk_index': idx,
            'text': ch,
            'search_text': preprocess_text(f"{doc['title']} | {doc['doc_type']} | {ch}"),
        })

chunks_df = pd.DataFrame(rows)
print(f"Total chunks: {len(chunks_df)}  |  from {chunks_df['document_id'].nunique()} documents")
chunks_df.groupby('document_id').size().rename('n_chunks')

Total chunks: 391  |  from 19 documents


,n_chunks
document_id,
D1,8
D10,40
D11,37
D12,14
D13,40
D14,21
D15,10
D16,11
D17,38


## Section 3 — Building Retrievers (Task 3)

Four retrievers are built:

| Retriever | Idea | Required by Task 3? |
|---|---|---|
| **TF-IDF** | Lexical baseline — exact word/phrase overlap, with bigrams. | Yes |
| **BM25** | Stronger lexical model — term-frequency saturation + length normalization. | Bonus (Lab 6/7 material) |
| **Semantic Embeddings** | `SentenceTransformer("all-MiniLM-L6-v2")` dense vectors capture *meaning*, so `"gum disease"` can match a chunk that only says `"periodontal disease"`. | Yes |
| **Hybrid** | `hybrid_score = alpha * semantic_score + (1 - alpha) * lexical_score`, both min-max normalized to [0, 1]. | Yes |

> **Environment note:** this authoring sandbox has no internet access to Hugging Face, so the
> code below tries to download `all-MiniLM-L6-v2` first (this is exactly what will happen in your
> Colab/Jupyter environment, which does have internet) and automatically falls back to a
> lightweight TF-IDF + LSA (`TruncatedSVD`) pseudo-embedding *only* so the notebook still runs
> end-to-end here. When you run this notebook in Colab, `SEMANTIC_MODE` will print
> `sentence-transformers` and you will get real semantic search — that is the mode that should be
> used for grading.

In [23]:
corpus = chunks_df['search_text'].tolist()

# --- TF-IDF ---
tfidf_vec = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
tfidf_matrix = tfidf_vec.fit_transform(corpus)

def tfidf_search(query, top_k=8):
    q_vec = tfidf_vec.transform([preprocess_text(query)])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    order = np.argsort(-scores)[:top_k]
    return [(chunks_df.iloc[i]['chunk_id'], float(scores[i])) for i in order]

In [24]:
# --- BM25 (bonus retriever) ---
tokenized_corpus = [c.lower().split() for c in corpus]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, top_k=8):
    scores = bm25.get_scores(preprocess_text(query).lower().split())
    order = np.argsort(-scores)[:top_k]
    return [(chunks_df.iloc[i]['chunk_id'], float(scores[i])) for i in order]

In [25]:
# --- Semantic Embeddings (real model, with offline fallback for this sandbox) ---
SEMANTIC_MODE = None
try:
    from sentence_transformers import SentenceTransformer
    st_model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = st_model.encode(corpus, show_progress_bar=False)
    SEMANTIC_MODE = "sentence-transformers"
except Exception as e:
    from sklearn.decomposition import TruncatedSVD
    svd_model = TruncatedSVD(n_components=64, random_state=42)
    embeddings = svd_model.fit_transform(tfidf_matrix)
    SEMANTIC_MODE = "lsa_fallback"

print("SEMANTIC_MODE:", SEMANTIC_MODE)

def semantic_search(query, top_k=8):
    if SEMANTIC_MODE == "sentence-transformers":
        q_emb = st_model.encode([query])
    else:
        q_emb = svd_model.transform(tfidf_vec.transform([preprocess_text(query)]))
    scores = cosine_similarity(q_emb, embeddings).flatten()
    order = np.argsort(-scores)[:top_k]
    return [(chunks_df.iloc[i]['chunk_id'], float(scores[i])) for i in order]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SEMANTIC_MODE: sentence-transformers


In [26]:
# --- Hybrid (lexical + semantic) ---
def normalize(scores):
    scores = np.asarray(scores, dtype=float)
    if scores.max() - scores.min() < 1e-9:
        return np.zeros_like(scores)
    return (scores - scores.min()) / (scores.max() - scores.min())

def hybrid_search(query, top_k=8, alpha=0.6):
    q_tfidf = tfidf_vec.transform([preprocess_text(query)])
    lexical_scores = cosine_similarity(q_tfidf, tfidf_matrix).flatten()
    if SEMANTIC_MODE == "sentence-transformers":
        q_emb = st_model.encode([query])
    else:
        q_emb = svd_model.transform(q_tfidf)
    semantic_scores = cosine_similarity(q_emb, embeddings).flatten()

    hybrid_scores = alpha * normalize(semantic_scores) + (1 - alpha) * normalize(lexical_scores)
    order = np.argsort(-hybrid_scores)[:top_k]
    return [(chunks_df.iloc[i]['chunk_id'], float(hybrid_scores[i])) for i in order]

In [27]:
# quick sanity check across all four retrievers, including one paraphrase-trap query
q = "How do I know if my gum disease treatment is actually working?"
for name, fn in [("TF-IDF", tfidf_search), ("BM25", bm25_search),
                  ("Semantic", semantic_search), ("Hybrid", hybrid_search)]:
    print(f"\n{name}:")
    for cid, score in fn(q, top_k=5):
        row = chunks_df.loc[chunks_df.chunk_id == cid].iloc[0]
        flag = "CURRENT" if row.is_current else "OUTDATED"
        print(f"  {cid:8s} (doc={row.document_id:4s} {flag:8s})  score={score:.3f}")


TF-IDF:
  D2_10    (doc=D2   OUTDATED)  score=0.276
  D2_0     (doc=D2   OUTDATED)  score=0.254
  D2_5     (doc=D2   OUTDATED)  score=0.215
  D2_11    (doc=D2   OUTDATED)  score=0.207
  D2_12    (doc=D2   OUTDATED)  score=0.187

BM25:
  D2_5     (doc=D2   OUTDATED)  score=10.549
  D2_11    (doc=D2   OUTDATED)  score=9.528
  D2_12    (doc=D2   OUTDATED)  score=8.922
  D2_4     (doc=D2   OUTDATED)  score=8.625
  D2_0     (doc=D2   OUTDATED)  score=8.564

Semantic:
  D2_12    (doc=D2   OUTDATED)  score=0.564
  D2_7     (doc=D2   OUTDATED)  score=0.564
  D2_3     (doc=D2   OUTDATED)  score=0.561
  D2_11    (doc=D2   OUTDATED)  score=0.560
  D13_38   (doc=D13  CURRENT )  score=0.551

Hybrid:
  D2_10    (doc=D2   OUTDATED)  score=0.974
  D2_0     (doc=D2   OUTDATED)  score=0.896
  D2_11    (doc=D2   OUTDATED)  score=0.894
  D2_12    (doc=D2   OUTDATED)  score=0.871
  D2_5     (doc=D2   OUTDATED)  score=0.843


## Section 4 — Queries and Ground Truth (Task 2)

14 queries are defined (more than the required 10), each with a manually verified ground-truth
list of relevant `document_id`s. The `is_paraphrase` column marks queries whose wording
deliberately differs from the document's own vocabulary (7 of the 14, well over the required 4):

| # | Paraphrase trap | Query wording | Document wording |
|---|---|---|---|
| 1 | Yes | "having a tooth pulled" | "tooth extraction" |
| 3 | Yes | "can't open my mouth all the way" | "Trismus (stiffness)" |
| 6 | Yes | "drink coffee while my aligner trays are in" | "Drink dark teas, coffee or soda with trays in" |
| 7 | Yes | "missing spots when I brush" | "areas you might be missing" |
| 10 | Yes | "gum disease treatment ... working" | current doc says "periodontal disease"; only the *outdated* doc says "gum disease" |
| 11 | Yes | "sleep with my new dentures" | "DO NOT wear ... dentures to bed" |
| 12 | Yes | "night guard doesn't fit right" | "If your night guard feels uncomfortable" |

Query #2 and #10 are also **current-vs-outdated conflict probes**: #2 hits documents that share
near-duplicate boilerplate (irrigation syringe instructions repeated almost verbatim across
several current documents), and #10 is deliberately the hardest case in the set — a lexical
retriever can be fooled into ranking the *outdated* `D2` above the *current* `D12` simply because
`D2` happens to use the same words as the query.

In [28]:
GROUND_TRUTH = {
    "How do I control bleeding after having a tooth pulled?": ["D17", "D4"],
    "How do I use my irrigation syringe to keep the surgical site clean?": ["D7", "D11", "D17"],
    "Why can't I open my mouth all the way after my surgery?": ["D7", "D11"],
    "How should I take care of my new crown or bridge?": ["D9"],
    "How many hours a day should I wear my clear aligner trays?": ["D14"],
    "Can I drink coffee while my aligner trays are in?": ["D14"],
    "What's a good way to check if I'm missing spots when I brush my teeth?": ["D8"],
    "How long should I wear my take-home whitening gel tray each day?": ["D6", "D19"],
    "When can I start rinsing with salt water after my root canal?": ["D5"],
    "How do I know if my gum disease treatment is actually working?": ["D12"],
    "Is it okay to sleep with my new dentures in?": ["D16"],
    "What should I do if my night guard doesn't fit right anymore?": ["D15"],
    "What foods should I avoid right after deep cleaning (scaling and root planing)?": ["D12"],
    "How long do I need to avoid smoking after a bone graft?": ["D7"],
}

IS_PARAPHRASE = {
    "How do I control bleeding after having a tooth pulled?": True,
    "How do I use my irrigation syringe to keep the surgical site clean?": False,
    "Why can't I open my mouth all the way after my surgery?": True,
    "How should I take care of my new crown or bridge?": False,
    "How many hours a day should I wear my clear aligner trays?": False,
    "Can I drink coffee while my aligner trays are in?": True,
    "What's a good way to check if I'm missing spots when I brush my teeth?": True,
    "How long should I wear my take-home whitening gel tray each day?": False,
    "When can I start rinsing with salt water after my root canal?": False,
    "How do I know if my gum disease treatment is actually working?": True,
    "Is it okay to sleep with my new dentures in?": True,
    "What should I do if my night guard doesn't fit right anymore?": True,
    "What foods should I avoid right after deep cleaning (scaling and root planing)?": False,
    "How long do I need to avoid smoking after a bone graft?": False,
}

queries_df = pd.DataFrame({
    "query": list(GROUND_TRUTH.keys()),
    "relevant_docs": list(GROUND_TRUTH.values()),
    "is_paraphrase": [IS_PARAPHRASE[q] for q in GROUND_TRUTH],
})
print(f"Total queries: {len(queries_df)}  |  paraphrase-trap queries: {queries_df['is_paraphrase'].sum()}")
queries_df

Total queries: 14  |  paraphrase-trap queries: 7


,query,relevant_docs,is_paraphrase
0,How do I control bleeding after having a tooth pulled?,"[D17, D4]",True
1,How do I use my irrigation syringe to keep the surgical site clean?,"[D7, D11, D17]",False
2,Why can't I open my mouth all the way after my surgery?,"[D7, D11]",True
3,How should I take care of my new crown or bridge?,[D9],False
4,How many hours a day should I wear my clear aligner trays?,[D14],False
5,Can I drink coffee while my aligner trays are in?,[D14],True
6,What's a good way to check if I'm missing spots when I brush my teeth?,[D8],True
7,How long should I wear my take-home whitening gel tray each day?,"[D6, D19]",False
8,When can I start rinsing with salt water after my root canal?,[D5],False
9,How do I know if my gum disease treatment is actually working?,[D12],True


## Section 5 — Retrieval Metrics and Evaluation (Task 3)

Retrieval is evaluated **at the document level** (chunks from the same document are de-duplicated
by rank) using the standard top-`K=3` metrics taught in Lab 6:

- **Precision@K** — of the top K retrieved documents, what fraction are relevant?
- **Recall@K** — of all relevant documents, what fraction were retrieved in the top K?
- **Hit Rate@K** — did at least one relevant document appear in the top K? (0 or 1 per query)
- **MRR (Mean Reciprocal Rank)** — how high up was the *first* relevant document ranked?

In [29]:
K = 3

def precision_at_k(retrieved, relevant, k=K):
    top = retrieved[:k]
    return sum(d in relevant for d in top) / k

def recall_at_k(retrieved, relevant, k=K):
    top = retrieved[:k]
    return sum(d in relevant for d in top) / len(relevant) if relevant else 0.0

def hit_rate_at_k(retrieved, relevant, k=K):
    return 1.0 if any(d in relevant for d in retrieved[:k]) else 0.0

def mrr_at_k(retrieved, relevant, k=K):
    for rank, d in enumerate(retrieved[:k], start=1):
        if d in relevant:
            return 1.0 / rank
    return 0.0

def retrieved_doc_ids(search_fn, query, pool_k=K * 4):
    """Pull a larger pool of chunks, then de-duplicate to unique documents (rank-preserving)."""
    results = search_fn(query, top_k=pool_k)
    seen, doc_ids = set(), []
    for cid, _ in results:
        d = chunks_df.loc[chunks_df.chunk_id == cid, 'document_id'].values[0]
        if d not in seen:
            seen.add(d)
            doc_ids.append(d)
    return doc_ids

def evaluate_retriever(search_fn, name, **kwargs):
    metrics = {"precision": [], "recall": [], "hit_rate": [], "mrr": []}
    per_query = {}
    for query, relevant in GROUND_TRUTH.items():
        results = search_fn(query, top_k=K * 4, **kwargs) if kwargs else search_fn(query, top_k=K * 4)
        seen, doc_ids = set(), []
        for cid, _ in results:
            d = chunks_df.loc[chunks_df.chunk_id == cid, 'document_id'].values[0]
            if d not in seen:
                seen.add(d)
                doc_ids.append(d)
        metrics["precision"].append(precision_at_k(doc_ids, relevant))
        metrics["recall"].append(recall_at_k(doc_ids, relevant))
        metrics["hit_rate"].append(hit_rate_at_k(doc_ids, relevant))
        metrics["mrr"].append(mrr_at_k(doc_ids, relevant))
        per_query[query] = doc_ids
    summary = {k: np.mean(v) for k, v in metrics.items()}
    summary["retriever"] = name
    return summary, per_query

tfidf_summary, tfidf_per_query = evaluate_retriever(tfidf_search, "TF-IDF")
bm25_summary, bm25_per_query = evaluate_retriever(bm25_search, "BM25")
semantic_summary, semantic_per_query = evaluate_retriever(semantic_search, "Semantic")
hybrid_summary, hybrid_per_query = evaluate_retriever(hybrid_search, "Hybrid (alpha=0.6)")

results_summary = pd.DataFrame([tfidf_summary, bm25_summary, semantic_summary, hybrid_summary]).set_index("retriever").round(3)
results_summary

,precision,recall,hit_rate,mrr
retriever,,,,
TF-IDF,0.310,0.726,0.786,0.750
BM25,0.310,0.738,0.786,0.738
Semantic,0.357,0.833,0.929,0.821
Hybrid (alpha=0.6),0.333,0.762,0.857,0.750


### Alpha tuning for the hybrid retriever (Task 3 requirement: test at least 3 alpha values)

`alpha` controls how much weight the semantic score gets versus the lexical (TF-IDF) score:
`hybrid_score = alpha * semantic + (1 - alpha) * lexical`. We test `alpha in {0.3, 0.5, 0.7}` and
report which value produces the best average metric score.

In [30]:
alpha_rows = []
for alpha in [0.3, 0.5, 0.7]:
    summary, _ = evaluate_retriever(hybrid_search, f"Hybrid (alpha={alpha})", alpha=alpha)
    alpha_rows.append(summary)

alpha_results = pd.DataFrame(alpha_rows).set_index("retriever").round(3)
alpha_results["average"] = alpha_results.mean(axis=1)
best_alpha_row = alpha_results["average"].idxmax()
print(f"Best alpha configuration on this dataset: {best_alpha_row}")
alpha_results

Best alpha configuration on this dataset: Hybrid (alpha=0.7)


,precision,recall,hit_rate,mrr,average
retriever,,,,,
Hybrid (alpha=0.3),0.333,0.762,0.857,0.738,0.67250
Hybrid (alpha=0.5),0.333,0.762,0.857,0.750,0.67550
Hybrid (alpha=0.7),0.357,0.833,0.929,0.786,0.72625


**How to read this table:** the alpha value with the highest `average` column is the one that
best balances lexical precision (good for exact terms like "Invisalign", "Opalescence", drug
names) against semantic recall (good for paraphrases like "gum disease" -> "periodontal disease").
A higher alpha leans more semantic; a lower alpha leans more lexical. Re-run this cell once
`SEMANTIC_MODE` prints `sentence-transformers` in your own environment, since the LSA fallback
used in this sandbox is a much weaker approximation of real semantic similarity than the actual
sentence-transformer model.

## Section 6 — Context Building (Task 4)

Raw candidate chunks from the hybrid retriever are candidate evidence, **not** final context.
`build_context()` applies four steps, in order:

1. **Current/outdated conflict resolution** — if retrieved chunks share the same `doc_type` but
   come from different documents (e.g. the old and new crown/bridge handouts, or the old and new
   scaling/root-planing handouts), only the chunk(s) from the document marked `is_current=True`
   are kept. This is the safeguard that stops the outdated `D1`/`D2` handouts from ever reaching
   the final context, even if a lexical retriever accidentally scores them higher.
2. **Outdated safety net** — any remaining chunk whose document is `is_current=False` is dropped
   outright, even if it wasn't part of a same-`doc_type` conflict.
3. **Near-duplicate chunk removal** — several current documents in this real dataset share almost
   identical boilerplate paragraphs (the "IRRIGATION SYRINGE", "BLEEDING", "SMOKING" sections are
   nearly word-for-word the same across the bone graft, implant placement, gum graft and
   extraction handouts). Keeping every copy would waste the word budget on redundant text, so
   chunks whose text is >=85% similar (`difflib.SequenceMatcher` ratio) to a chunk already kept
   are dropped, keeping only the highest-scoring copy.
4. **150-word budget** — chunks are added, highest score first, until adding the next one would
   exceed 150 words.

Each surviving chunk is labeled with its title, `effective_date`, and a `CURRENT`/`OUTDATED` tag
for the prompt.

In [31]:
def is_near_duplicate(text_a, text_b, threshold=0.85):
    return SequenceMatcher(None, text_a, text_b).ratio() >= threshold

def build_context(query, pool_size=10, alpha=0.6, min_score=0.02, max_words=150):
    candidates = hybrid_search(query, top_k=pool_size, alpha=alpha)

    evidence = []
    for cid, score in candidates:
        if score < min_score:
            continue
        row = chunks_df.loc[chunks_df.chunk_id == cid].iloc[0]
        evidence.append({
            "chunk_id": cid, "document_id": row.document_id, "title": row.title,
            "doc_type": row.doc_type, "effective_date": row.effective_date,
            "is_current": bool(row.is_current), "text": row.text, "score": float(score),
        })

    # Step 1: same doc_type, multiple documents -> keep only the is_current one
    by_type = {}
    for e in evidence:
        by_type.setdefault(e["doc_type"], []).append(e)
    filtered = []
    for doc_type, items in by_type.items():
        docs_present = {i["document_id"] for i in items}
        if len(docs_present) > 1:
            current_items = [i for i in items if i["is_current"]]
            keep_pool = current_items if current_items else items
            keep_doc = max(keep_pool, key=lambda x: x["effective_date"])["document_id"]
            items = [i for i in items if i["document_id"] == keep_doc]
        filtered.extend(items)

    # Step 2: outdated safety net
    filtered = [e for e in filtered if e["is_current"]]

    # Step 3: near-duplicate chunk removal (highest score wins)
    filtered = sorted(filtered, key=lambda x: -x["score"])
    deduped = []
    for e in filtered:
        if any(is_near_duplicate(e["text"], kept["text"]) for kept in deduped):
            continue
        deduped.append(e)

    # Step 4: 150-word budget
    final, used_words = [], 0
    for e in deduped:
        n_words = len(e["text"].split())
        if used_words + n_words > max_words and final:
            break
        final.append(e)
        used_words += n_words

    return final

def format_context_package(evidence):
    lines = []
    for i, e in enumerate(evidence, start=1):
        status = "CURRENT" if e["is_current"] else "OUTDATED"
        lines.append(f"[{i}] Source: {e['title']} ({status}, updated {e['effective_date']})\n{e['text']}")
    return "\n\n".join(lines)

### Build context packages for at least 5 queries (Task 4), including the current/outdated conflict

In [33]:
context_demo_queries = [
    "How should I take care of my new crown or bridge?",          # crown/bridge conflict
    "How do I know if my gum disease treatment is actually working?",  # scaling conflict + paraphrase trap
    "How do I use my irrigation syringe to keep the surgical site clean?",  # near-duplicate chunks
    "How many hours a day should I wear my clear aligner trays?",
    "When can I start rinsing with salt water after my root canal?",
]

context_packages = {}
for q in context_demo_queries:
    ctx = build_context(q)
    context_packages[q] = ctx
    total_words = sum(len(e['text'].split()) for e in ctx)
    print(f"QUERY: {q}")
    if ctx:
        print(pd.DataFrame(ctx)[['chunk_id', 'document_id', 'is_current', 'effective_date', 'score']].to_string(index=False))
    else:
        print("  No context chunks found after filtering.")
    print(f"context word count: {total_words} / 150\n{'-'*90}")

QUERY: How should I take care of my new crown or bridge?
chunk_id document_id  is_current effective_date    score
    D9_0          D9        True     2025-08-01 0.824999
   D9_36          D9        True     2025-08-01 0.768666
   D9_35          D9        True     2025-08-01 0.712269
   D9_12          D9        True     2025-08-01 0.676858
context word count: 127 / 150
------------------------------------------------------------------------------------------
QUERY: How do I know if my gum disease treatment is actually working?
  No context chunks found after filtering.
context word count: 0 / 150
------------------------------------------------------------------------------------------
QUERY: How do I use my irrigation syringe to keep the surgical site clean?
chunk_id document_id  is_current effective_date    score
    D9_5          D9        True     2025-08-01 0.997199
    D7_0          D7        True     2025-01-20 0.918043
    D9_4          D9        True     2025-08-01 0.905388
co

### The current/outdated conflict, made explicit

For **"How should I take care of my new crown or bridge?"**, both the outdated `D1` (2021) and
the current `D9` (2025) handout describe crown/bridge aftercare and share
`doc_type="crown_bridge_care"`. Inspect the raw candidate pool *before* `build_context()` filters
it, compared to the final context package:

In [34]:
q = "How should I take care of my new crown or bridge?"
raw_candidates = hybrid_search(q, top_k=10)
print("RAW candidates before context building:")
for cid, score in raw_candidates:
    row = chunks_df.loc[chunks_df.chunk_id == cid].iloc[0]
    flag = "CURRENT" if row.is_current else "OUTDATED"
    print(f"  {cid:8s} doc={row.document_id:4s} {flag:8s} score={score:.3f}")

print("\nFINAL context package after conflict resolution (only CURRENT D9 survives):")
for e in context_packages[q]:
    print(f"  {e['chunk_id']:8s} doc={e['document_id']:4s} CURRENT score={e['score']:.3f}")

RAW candidates before context building:
  D1_0     doc=D1   OUTDATED score=1.000
  D9_0     doc=D9   CURRENT  score=0.825
  D9_36    doc=D9   CURRENT  score=0.769
  D9_35    doc=D9   CURRENT  score=0.712
  D9_12    doc=D9   CURRENT  score=0.677
  D9_4     doc=D9   CURRENT  score=0.672
  D9_15    doc=D9   CURRENT  score=0.660
  D1_5     doc=D1   OUTDATED score=0.652
  D9_23    doc=D9   CURRENT  score=0.642
  D9_6     doc=D9   CURRENT  score=0.636

FINAL context package after conflict resolution (only CURRENT D9 survives):
  D9_0     doc=D9   CURRENT score=0.825
  D9_36    doc=D9   CURRENT score=0.769
  D9_35    doc=D9   CURRENT score=0.712
  D9_12    doc=D9   CURRENT score=0.677


## Section 7 — Writing Three Prompts (Task 5)

Same context package, three prompt styles with increasing structure, following the prompt anatomy
taught in Lab 8 (role, task, evidence boundary, operational rules, output format):

1. **Weak prompt** — just dumps the context and question. No rules, no grounding constraint, no
   citation requirement. The model is free to hallucinate or ramble.
2. **Better prompt** — adds grounding rules and a citation requirement, but the output is still
   free-form prose.
3. **Strict prompt** — full anatomy: role, task, evidence boundary, operational rules (cite
   sources, refuse if unsupported, resolve conflicts by recency), and a **structured two-part
   output format** (`ANSWER:` then `SOURCES USED:`).

**When each style is the wrong choice:**

- **Weak** is the wrong choice for this domain almost always — patient-safety questions need
  grounding, so a weak prompt is really only defensible for very low-stakes, exploratory
  brainstorming where hallucination has no real consequence (e.g. "give me five witty names for
  our clinic newsletter").
- **Better** is the wrong choice when the answer needs to be machine-parsed downstream (e.g. fed
  into a patient app that displays "citations" as a separate structured field) — free-form prose
  with inline citation numbers is hard to parse reliably compared to a strict, fixed schema.
- **Strict** is the wrong choice when a patient asks a simple, anxious, low-information question
  (e.g. "is it going to hurt?") — a rigid two-part `ANSWER:` / `SOURCES USED:` format can feel
  cold and clinical exactly when a warmer, more conversational tone would reassure the patient
  better.

In [35]:
WEAK_PROMPT = """Answer the question using the context.
Context: {context}
Question: {question}"""

BETTER_PROMPT = """You are a dental patient-education assistant. Use the context below to answer
the patient's question. Only use information that is actually stated in the context, and mention
which source number(s) support each claim, like [1].

Context:
{context}

Question: {question}

Answer:"""

STRICT_PROMPT = """You are a dental patient-education assistant.

Task: Answer the patient's question using ONLY the evidence in the context package below.

Rules:
- If the answer is not fully supported by the context, say you don't have enough information and recommend they call the clinic.
- Every source in the context package is already labeled CURRENT or OUTDATED. Never base your answer on an OUTDATED source; if only an OUTDATED source is present, say so explicitly.
- If two CURRENT sources conflict, prefer the most recently updated one and note the conflict.
- Do not give medical advice beyond what is written in the sources.
- Keep the answer under 150 words, in plain, patient-friendly language.

Context package:
{context}

Patient question: {question}

Respond in exactly this two-part format:
ANSWER: <your grounded answer, with inline citations like [1]>
SOURCES USED: <comma-separated list of the source numbers you actually relied on>"""

question = "How should I take care of my new crown or bridge?"
context_text = format_context_package(context_packages[question])

weak_prompt = WEAK_PROMPT.format(context=context_text, question=question)
better_prompt = BETTER_PROMPT.format(context=context_text, question=question)
strict_prompt = STRICT_PROMPT.format(context=context_text, question=question)

print(strict_prompt)

You are a dental patient-education assistant.

Task: Answer the patient's question using ONLY the evidence in the context package below.

Rules:
- If the answer is not fully supported by the context, say you don't have enough information and recommend they call the clinic.
- Every source in the context package is already labeled CURRENT or OUTDATED. Never base your answer on an OUTDATED source; if only an OUTDATED source is present, say so explicitly.
- If two CURRENT sources conflict, prefer the most recently updated one and note the conflict.
- Do not give medical advice beyond what is written in the sources.
- Keep the answer under 150 words, in plain, patient-friendly language.

Context package:
[1] Source: Crown or Bridge Post-Operative Care (Updated) (CURRENT, updated 2025-08-01)
CROWN OR BRIDGE - DO NOT DISTURB THE AREA: For the next few days, and especially the first 24 hours, it is very important to allow your body to form a good clot and start natural healing process.

[2] So

## Section 8 — First LLM Answer (Lab 9 style)

The strict, two-part prompt is sent to the LLM. `call_llm()` calls the **Anthropic API** if an
`ANTHROPIC_API_KEY` is set in the environment (works out of the box on your own machine/Colab
with `pip install anthropic`); otherwise it falls back to a clearly-labeled extractive summary
built directly from the top context chunks, so the notebook still runs end-to-end for
demonstration/grading purposes without a key.

In [36]:
def call_llm(prompt, model="claude-sonnet-4-5", max_tokens=300):
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if api_key:
        try:
            import anthropic
            client = anthropic.Anthropic(api_key=api_key)
            resp = client.messages.create(
                model=model, max_tokens=max_tokens,
                messages=[{"role": "user", "content": prompt}],
            )
            return "".join(b.text for b in resp.content if b.type == "text")
        except Exception as e:
            return f"[LLM call failed: {e}]"
    # --- offline fallback: simple extractive answer from the context, for demo only ---
    evidence_block = prompt.split("Context package:")[-1].split("Patient question:")[0].strip()
    bullets = [f"- {line.strip()}" for line in evidence_block.split(". ") if line.strip()]
    return ("[SIMULATED ANSWER -- no ANTHROPIC_API_KEY set in this environment]\n"
            "ANSWER: Based on the retrieved CURRENT sources, here is a grounded extractive summary:\n"
            + "\n".join(bullets[:8])
            + "\nSOURCES USED: see chunk_id list printed above")

for q in [
    "How should I take care of my new crown or bridge?",
    "How do I know if my gum disease treatment is actually working?",
    "How many hours a day should I wear my clear aligner trays?",
]:
    ctx = build_context(q)
    prompt = STRICT_PROMPT.format(context=format_context_package(ctx), question=q)
    answer = call_llm(prompt)
    print(f"Q: {q}\n{answer}\n{'-'*90}")

Q: How should I take care of my new crown or bridge?
[SIMULATED ANSWER -- no ANTHROPIC_API_KEY set in this environment]
ANSWER: Based on the retrieved CURRENT sources, here is a grounded extractive summary:
- [1] Source: Crown or Bridge Post-Operative Care (Updated) (CURRENT, updated 2025-08-01)
CROWN OR BRIDGE - DO NOT DISTURB THE AREA: For the next few days, and especially the first 24 hours, it is very important to allow your body to form a good clot and start natural healing process.

[2] Source: Crown or Bridge Post-Operative Care (Updated) (CURRENT, updated 2025-08-01)
have significantly more discomfort, and the success of the procedure may be affected.

[3] Source: Crown or Bridge Post-Operative Care (Updated) (CURRENT, updated 2025-08-01)
check
- Following these instructions very closely will greatly help your comfort, and promote uneventful healing of the area
- If any of the instructions are not followed, you might have significantly more discomfort, and the success of the pr

## Section 9 — Error Analysis (Task 6)

Rather than hand-picking failures, this section **computes them automatically** from the
evaluation already run in Section 5, exactly the way Lab 6 (Section 45), Lab 7 (Task 7) and
Lab 8 (Task 6) all ask you to: compare what each retriever actually returned against the
ground truth, and classify *why* it failed.

For every query where the Hybrid retriever's top-3 documents do not fully cover the ground truth,
the cell below builds a diagnostic row with:
- the query and its correct document(s),
- what Hybrid, TF-IDF and Semantic each returned instead,
- a rule-based guess at which pipeline layer is responsible (retrieval vocabulary/ranking gap vs.
  context-assembly over-filtering vs. generation), and
- a suggested fix.

**Run this notebook with real internet access (`SEMANTIC_MODE == "sentence-transformers"`)
before finalizing the written analysis below** — the LSA fallback used in this offline sandbox is
a much weaker stand-in for real embeddings, so the exact set of failing queries may change once
real semantic search is active. Use the printed table as the source of truth for the 3+ write-ups
required by Task 6.

In [37]:
def diagnose_failure(query, relevant):
    hybrid_docs = hybrid_per_query[query]
    tfidf_docs = tfidf_per_query[query]
    semantic_docs = semantic_per_query[query]

    hybrid_hit = hit_rate_at_k(hybrid_docs, relevant)
    if hybrid_hit == 1.0:
        return None  # not a failure for the retriever we actually ship (hybrid)

    found_anywhere = any(d in tfidf_docs[:K*4] or d in semantic_docs[:K*4] for d in relevant)
    if not found_anywhere:
        layer = "retrieval (vocabulary/embedding gap: no retriever's candidate pool contained the correct document)"
        fix = "Add the missing vocabulary to search_text (e.g. include synonyms in doc_type/title), or enrich chunk metadata."
    elif any(d in tfidf_docs[:K] for d in relevant) and not any(d in hybrid_docs[:K] for d in relevant):
        layer = "retrieval (alpha weighting: TF-IDF ranked it in the top 3 but the hybrid blend pushed it out)"
        fix = "Lower alpha (lean more lexical) for this query type, or re-check the alpha sweep in Section 5."
    elif any(d in semantic_docs[:K] for d in relevant) and not any(d in hybrid_docs[:K] for d in relevant):
        layer = "retrieval (alpha weighting: Semantic ranked it in the top 3 but the hybrid blend pushed it out)"
        fix = "Raise alpha (lean more semantic) for this query type."
    else:
        layer = "retrieval (ranking: found in a larger pool but not within top 3 by any single retriever)"
        fix = "Increase pool_size / top_k before re-ranking, or add a metadata boost for exact-match doc_type."

    return {
        "query": query,
        "correct_document(s)": relevant,
        "hybrid_returned_instead": hybrid_docs[:K],
        "tfidf_returned": tfidf_docs[:K],
        "semantic_returned": semantic_docs[:K],
        "likely_failing_layer": layer,
        "suggested_fix": fix,
    }

failure_rows = []
for query, relevant in GROUND_TRUTH.items():
    diag = diagnose_failure(query, relevant)
    if diag is not None:
        failure_rows.append(diag)

failure_df = pd.DataFrame(failure_rows)
print(f"Number of queries where Hybrid@3 missed the ground truth: {len(failure_df)}")
failure_df

Number of queries where Hybrid@3 missed the ground truth: 2


,query,correct_document(s),hybrid_returned_instead,tfidf_returned,semantic_returned,likely_failing_layer,suggested_fix
0,What's a good way to check if I'm missing spots when I brush my teeth?,[D8],"[D11, D9, D7]","[D10, D11, D7]","[D8, D16, D12]",retrieval (alpha weighting: Semantic ranked it in the top 3 but the hybrid blend pushed it out),Raise alpha (lean more semantic) for this query type.
1,How do I know if my gum disease treatment is actually working?,[D12],[D2],[D2],"[D2, D13, D5]",retrieval (vocabulary/embedding gap: no retriever's candidate pool contained the correct document),"Add the missing vocabulary to search_text (e.g. include synonyms in doc_type/title), or enrich c..."


### Context-assembly and generation-layer checks

The retrieval-level failures above cover the "wrong evidence was retrieved" case. Two more
failure modes are worth checking explicitly, per the four-way classification used in Lab 9
(chunking / retrieval / context assembly / generation):

- **Context-assembly failure**: the correct document *was* retrieved, but `build_context()`
  dropped it anyway (e.g. the near-duplicate filter or the 150-word budget cut it before it was
  added). Check this by comparing `hybrid_per_query[query]` against the `document_id`s that
  actually appear in `context_packages[query]`.
- **Generation failure**: the context package contains the right evidence, but `call_llm()`'s
  answer still doesn't use it (only checkable once `ANTHROPIC_API_KEY` is set and you are getting
  real generations instead of the offline extractive fallback).

In [38]:
for q in context_demo_queries:
    retrieved_docs = {chunks_df.loc[chunks_df.chunk_id == cid, 'document_id'].values[0]
                      for cid, _ in hybrid_search(q, top_k=K * 4)}
    context_docs = {e['document_id'] for e in context_packages[q]}
    dropped = retrieved_docs - context_docs
    relevant = set(GROUND_TRUTH.get(q, []))
    dropped_relevant = dropped & relevant
    if dropped_relevant:
        print(f"CONTEXT-ASSEMBLY FLAG for '{q}':")
        print(f"  relevant doc(s) retrieved but dropped from final context: {dropped_relevant}")
        print(f"  (check whether this was the outdated-safety-net, the near-duplicate filter, or the 150-word budget)\n")

CONTEXT-ASSEMBLY FLAG for 'How do I use my irrigation syringe to keep the surgical site clean?':
  relevant doc(s) retrieved but dropped from final context: {'D11', 'D17'}
  (check whether this was the outdated-safety-net, the near-duplicate filter, or the 150-word budget)



### Written error analysis (fill in from the actual printed output above once run with real embeddings)

**Failure 1**
1. Query: *(copy the first row's `query` from `failure_df`)*
2. Correct document: *(copy `correct_document(s)`)*
3. What the retriever returned instead: *(copy `hybrid_returned_instead`)*
4. Failing layer: *(copy `likely_failing_layer`)*
5. Fix: *(copy `suggested_fix`)*

**Failure 2** — repeat using the second row of `failure_df` (or a context-assembly flag above, if
any query triggered the near-duplicate/word-budget check).

**Failure 3** — repeat using the third row of `failure_df`, or, if `failure_df` has fewer than 3
rows once real embeddings are used, pick a query with the lowest `hit_rate` in `results_summary`
for TF-IDF alone, and explain why the semantic/hybrid retriever fixes it (this is exactly the
"gum disease" vs. "periodontal disease" paraphrase case if it appears here).

## Summary

This notebook implements the full pipeline required by the Lab 8 Final Assignment, run end-to-end
on a real, self-collected knowledge base of 19 dental patient-education documents:

**Documents -> Preprocessing -> Chunks -> Retriever (TF-IDF / BM25 / Semantic / Hybrid, alpha-tuned)
-> Candidate Evidence -> Context Building (current/outdated conflict resolution, near-duplicate
removal, 150-word budget) -> Context Package -> Prompt (weak -> better -> strict) -> LLM Answer
-> Error Analysis.**

Key things this real dataset demonstrated that a synthetic toy corpus would not:
- A genuine current-vs-outdated conflict on **two** topics (crown/bridge, scaling/root planing),
  including a case where the *outdated* document is lexically closer to the query wording than
  the *current* one (`"gum disease"` vs. `"periodontal disease"`) — a strong argument for why
  metadata-based filtering (`is_current`) must never be skipped in favor of raw retrieval score.
- Real near-duplicate boilerplate shared across multiple current documents (irrigation-syringe,
  bleeding, smoking, and pain-management paragraphs repeated almost verbatim across several
  post-surgical handouts), which made the near-duplicate dedup step in context building
  meaningfully useful rather than a formality.

### To finish grading this notebook
1. Run it in Google Colab (or any environment with real internet access) so
   `SEMANTIC_MODE` prints `sentence-transformers` instead of `lsa_fallback`.
2. Set `ANTHROPIC_API_KEY` in the Colab environment (`os.environ['ANTHROPIC_API_KEY'] = "..."`
   or use Colab secrets) to get real generated answers in Section 8 instead of the offline
   extractive fallback.
3. Re-check Section 9's `failure_df` output once real embeddings are active, and copy the 3
   required write-ups from the actual printed rows.